In [1]:
import sys
print(sys.executable)
print(sys.version)

c:\Users\celal\OneDrive\Desktop\ayna-hackathon-2026\ayna-hackathon-2026\notebooks\citypilot-env\Scripts\python.exe
3.13.1 (tags/v3.13.1:0671451, Dec  3 2024, 19:06:28) [MSC v.1942 64 bit (AMD64)]


In [3]:
from pathlib import Path
import sys

current = Path.cwd()
repo = next(
    (p for p in [current, *current.parents] if (p / ".git").exists()),
    None,
)

if repo is None:
    raise RuntimeError("Repo kökü tapılmadı.")

print("Python:", sys.executable)
print("Repo:", repo)

print("\nRepo daxilində:")
for path in sorted(repo.iterdir()):
    if not path.name.startswith("."):
        print("Qovluq:" if path.is_dir() else "Fayl:", path.name)

Python: c:\Users\celal\OneDrive\Desktop\ayna-hackathon-2026\ayna-hackathon-2026\notebooks\citypilot-env\Scripts\python.exe
Repo: c:\Users\celal\OneDrive\Desktop\ayna-hackathon-2026\ayna-hackathon-2026

Repo daxilində:
Qovluq: data
Qovluq: docs
Qovluq: notebooks
Qovluq: outputs
Fayl: README.md
Fayl: requirements.txt
Qovluq: src
Qovluq: tests
Qovluq: tools


In [4]:
data_dir = repo / "data"
data_files = sorted(p for p in data_dir.rglob("*") if p.is_file())

print(f"Data qovluğunda {len(data_files)} fayl var:\n")

for path in data_files[:50]:
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"{path.relative_to(data_dir)} — {size_mb:.2f} MB")

Data qovluğunda 1 fayl var:

.gitkeep — 0.00 MB


In [6]:
%pip install pandas

   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.6 MB ? eta -:--:--
   -- ------------------------------------- 0.5/9.6 MB 2.5 MB/s eta 0:00:04
   --- ------------------------------------ 0.8/9.6 MB 1.5 MB/s eta 0:00:06
   ---- ----------------------------------- 1.0/9.6 MB 1.7 MB/s eta 0:00:05
   ---- ----------------------------------- 1.0/9.6 MB 1.7 MB/s eta 0:00:05
   ----- ---------------------------------- 1.3/9.6 MB 1.1 MB/s eta 0:00:08
   ----- ---------------------------------- 1.3/9.6 MB 1.1 MB/s eta 0:00:08
   ------ --------------------------------- 1.6/9.6 MB 1.1 MB/s eta 0:00:08
   ------ --------------------------------- 1.6/9.6 MB 1.1 MB/s eta 0:00:08
   ------- -------------------------------- 1.8/9.6 MB 964.1 kB/s eta 0:00:09
   ------- -------------------------------- 1.8/9.6 MB 964.1 kB/s eta 0:00:09
   -------- -----------------

In [7]:
from pathlib import Path
from zipfile import ZipFile
import pandas as pd

ayna_zip = repo / "data" / "raw" / "AYNA_Datasets_43.zip"
osm_zip = repo / "data" / "raw" / "azerbaijan-261016-free.shp.zip"

with ZipFile(ayna_zip) as archive:
    with archive.open("AYNA_Datasets/05_metro_exits/pathways.csv") as file:
        metro_raw = pd.read_csv(file, sep=";", encoding="utf-8-sig")

print("Sətir və sütun sayı:", metro_raw.shape)
display(metro_raw.head())

Sətir və sütun sayı: (98, 6)


,Stansiya,Station,coğrafi enliyi (latitude),coğrafi uzunluğu (longitude),yerləşdiyi ünvan,Çıxış nörmələri
0,Xocəsən,Khojasan,40421499,49778066,Dairəvi yol,Çıxış 1
1,Avtovağzal,Avtovaghzal,40421760,49796061,Bakı Sumqayıt yolu,Çıxış 1
2,Avtovağzal,Avtovaghzal,40421344,49795113,Bakı Sumqayıt yolu,Çıxış 2
3,Avtovağzal,Avtovaghzal,40420451,49795884,Bakı Sumqayıt yolu,Çıxış 3
4,Avtovağzal,Avtovaghzal,40420312,49795509,Bakı Sumqayıt yolu,Çıxış 4


In [8]:
%pip install geopandas

  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.8 MB ? eta -:--:--
   ---------------------------------------- 0.3/23.8 MB ? eta -:--:--
   -------------------

In [10]:
import geopandas as gpd

metro = metro_raw.rename(columns={
    "Stansiya": "station_name",
    "Station": "station_en",
    "coğrafi enliyi (latitude)": "lat",
    "coğrafi uzunluğu (longitude)": "lon",
    "yerləşdiyi ünvan": "address",
    "Çıxış nörmələri": "exit_name",
}).copy()

for column in ["station_name", "station_en", "address", "exit_name"]:
    metro[column]= metro[column].astype("string").str.strip()

# Mənbədə koordinatlar mikro-dərəcə ilə verilib.
metro["lat"] = pd.to_numeric(metro["lat"], errors="raise") / 1_000_000
metro["lon"] = pd.to_numeric(metro["lon"], errors="raise") / 1_000_000

valid = metro["lat"].between(-90, 90) & metro["lon"].between(-180, 180)
if not valid.all():
    raise ValueError(f"{(~valid).sum()} sətirdə koordinat boş və ya yanlışdır.")

metro_exits = gpd.GeoDataFrame(
    metro,
    geometry=gpd.points_from_xy(metro["lon"], metro["lat"]),
    crs="EPSG:4326",
)

print("Çıxış sayı:", len(metro_exits))
print("CRS:", metro_exits.crs)
display(metro_exits[["station_name", "exit_name", "lat", "lon", "geometry"]].head())

Çıxış sayı: 98
CRS: EPSG:4326


,station_name,exit_name,lat,lon,geometry
0,Xocəsən,Çıxış 1,40.421499,49.778066,POINT (49.77807 40.4215)
1,Avtovağzal,Çıxış 1,40.421760,49.796061,POINT (49.79606 40.42176)
2,Avtovağzal,Çıxış 2,40.421344,49.795113,POINT (49.79511 40.42134)
3,Avtovağzal,Çıxış 3,40.420451,49.795884,POINT (49.79588 40.42045)
4,Avtovağzal,Çıxış 4,40.420312,49.795509,POINT (49.79551 40.42031)


In [11]:
roads_path = (
    f"zip://{osm_zip.as_posix()}!gis_osm_roads_free_1.shp"
)

roads_raw = gpd.read_file(roads_path)

print("Yol obyekti sayı:", len(roads_raw))
print("CRS:", roads_raw.crs)
print("Həndəsə növləri:")
print(roads_raw.geom_type.value_counts())

display(
    roads_raw[
        ["osm_id", "fclass", "name", "oneway", "layer", "bridge", "tunnel"]
    ].head()
)

print("\nYol sinifləri:")
print(roads_raw["fclass"].value_counts())

Yol obyekti sayı: 349296
CRS: EPSG:4326
Həndəsə növləri:
LineString    349296
Name: count, dtype: int64


,osm_id,fclass,name,oneway,layer,bridge,tunnel
0,4721947,secondary_link,NaN,F,0,F,F
1,16604950,trunk,NaN,B,0,F,F
2,16606271,unclassified,NaN,B,0,F,F
3,16703123,trunk,Heydər Əliyev prospekti,F,0,F,F
4,17694186,primary,Nizami Gəncəvi,F,0,F,F



Yol sinifləri:
fclass
living_street     163862
service            60085
track              42412
residential        31408
footway            14780
tertiary            8806
unclassified        8596
secondary           4434
primary             4381
path                2297
trunk               2015
steps                958
trunk_link           799
track_grade4         731
primary_link         578
secondary_link       513
track_grade3         451
track_grade5         419
motorway             373
motorway_link        276
pedestrian           251
track_grade2         247
cycleway             228
track_grade1         198
tertiary_link        148
busway                39
unknown                8
bridleway              3
Name: count, dtype: int64


In [12]:
from shapely.geometry import box

METRIC_CRS = "EPSG:32639"

# Spesifikasiyadakı ilkin analiz qutusu.
core_area = gpd.GeoSeries(
    [box(49.77, 40.355, 49.90, 40.44)],
    crs="EPSG:4326",
).to_crs(METRIC_CRS)

# Buffer metrlə hesablanır.
routing_area_m = core_area.buffer(1500)
routing_area_osm = routing_area_m.to_crs(roads_raw.crs).iloc[0]

# Sahəyə toxunan yolları bütöv saxlayırıq; həndəsəni kəsmirik.
selected = roads_raw.geometry.intersects(routing_area_osm)
roads_m = roads_raw.loc[selected].copy().to_crs(METRIC_CRS)

metro_exits_m = metro_exits.to_crs(METRIC_CRS)

# Yol xətlərinin uzunluğu artıq metrlədir.
roads_m["length_m"] = roads_m.geometry.length

print("Ölkə üzrə yol obyekti:", len(roads_raw))
print("İş sahəsində yol obyekti:", len(roads_m))
print("Metrik CRS:", roads_m.crs)

display(
    roads_m[["osm_id", "fclass", "length_m", "oneway", "bridge", "tunnel"]].head()
)

Ölkə üzrə yol obyekti: 349296
İş sahəsində yol obyekti: 25020
Metrik CRS: EPSG:32639


,osm_id,fclass,length_m,oneway,bridge,tunnel
0,4721947,secondary_link,153.713831,F,F,F
45,20893261,trunk_link,275.647027,F,F,F
52,21066101,trunk,572.857067,F,F,F
53,21066110,trunk,112.847731,F,T,F
54,21066477,trunk,221.323066,F,F,F


In [13]:
street_classes = {
    "living_street", "residential", "service", "unclassified",
    "primary", "primary_link",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link",
}

# Track alt siniflərini də eyni qaydaya daxil edirik.
track_classes = {"track"} | {
    f"track_grade{i}" for i in range(1, 6)
}

walk_classes = street_classes | track_classes | {
    "footway", "pedestrian", "path", "steps", "cycleway", "bridleway",
}

drive_classes = street_classes | {
    "motorway", "motorway_link", "trunk", "trunk_link", "busway",
}

# Spesifikasiyaya görə default: footway/pedestrian üzərində sürmək bağlıdır.
ALLOW_BIKE_ON_FOOTWAYS = False

bike_classes = walk_classes - {"steps", "footway", "pedestrian"}

if ALLOW_BIKE_ON_FOOTWAYS:
    bike_classes |= {"footway", "pedestrian"}

roads_m["walk_ok"] = roads_m["fclass"].isin(walk_classes)
roads_m["bike_ok"] = roads_m["fclass"].isin(bike_classes)
roads_m["drive_ok"] = roads_m["fclass"].isin(drive_classes)

print("Rejimlər üzrə uyğun yol obyekti sayı:")
print(roads_m[["walk_ok", "bike_ok", "drive_ok"]].sum())

print("\nHeç bir rejimə daxil edilməyən siniflər:")
unused = ~roads_m[["walk_ok", "bike_ok", "drive_ok"]].any(axis=1)
print(roads_m.loc[unused, "fclass"].value_counts(dropna=False))

print("\nOneway dəyərləri:")
print(roads_m["oneway"].value_counts(dropna=False))

Rejimlər üzrə uyğun yol obyekti sayı:
walk_ok     24590
bike_ok     18905
drive_ok    18845
dtype: int64

Heç bir rejimə daxil edilməyən siniflər:
Series([], Name: count, dtype: int64)

Oneway dəyərləri:
oneway
B    20601
F     4416
T        3
Name: count, dtype: int64


In [14]:
oneway = roads_m["oneway"].astype("string").str.strip().str.upper()

if not oneway.isin(["B", "F", "T"]).all():
    raise ValueError("Naməlum və ya boş oneway dəyəri var.")

# Forward/reverse xəttin koordinat sırasına görədir.
forward_allowed = oneway.isin(["B", "F"])
reverse_allowed = oneway.isin(["B", "T"])

roads_m["walk_forward"] = roads_m["walk_ok"]
roads_m["walk_reverse"] = roads_m["walk_ok"]

roads_m["drive_forward"] = roads_m["drive_ok"] & forward_allowed
roads_m["drive_reverse"] = roads_m["drive_ok"] & reverse_allowed

cycleway = roads_m["fclass"].eq("cycleway")
roads_m["bike_forward"] = roads_m["bike_ok"] & (forward_allowed | cycleway)
roads_m["bike_reverse"] = roads_m["bike_ok"] & (reverse_allowed | cycleway)

direction_summary = pd.DataFrame({
    "forward": [
        roads_m["walk_forward"].sum(),
        roads_m["bike_forward"].sum(),
        roads_m["drive_forward"].sum(),
    ],
    "reverse": [
        roads_m["walk_reverse"].sum(),
        roads_m["bike_reverse"].sum(),
        roads_m["drive_reverse"].sum(),
    ],
}, index=["walk", "bike", "drive"])

display(direction_summary)

,forward,reverse
walk,24590,24590
bike,18902,14935
drive,18842,14448


In [15]:
geometry_checks = pd.Series({
    "Boş geometry (null)": roads_m.geometry.isna().sum(),
    "Boş həndəsə (empty)": roads_m.geometry.is_empty.sum(),
    "Etibarsız həndəsə": (~roads_m.geometry.is_valid).sum(),
    "Sıfır uzunluq": roads_m["length_m"].eq(0).sum(),
})

print("Həndəsə yoxlaması:")
print(geometry_checks)

print("\nHəndəsə növləri:")
print(roads_m.geom_type.value_counts())

print("\nSəviyyə, körpü və tunel kombinasiyaları:")
display(
    roads_m.groupby(
        ["layer", "bridge", "tunnel"],
        dropna=False,
    ).size().rename("road_count").reset_index()
)

Həndəsə yoxlaması:
Boş geometry (null)    0
Boş həndəsə (empty)    0
Etibarsız həndəsə      0
Sıfır uzunluq          0
dtype: int64

Həndəsə növləri:
LineString    25020
Name: count, dtype: int64

Səviyyə, körpü və tunel kombinasiyaları:


,layer,bridge,tunnel,road_count
0,-2,F,T,6
1,-1,F,F,14
2,-1,F,T,214
3,0,F,F,24599
4,0,F,T,45
5,0,T,F,1
6,1,F,F,10
7,1,T,F,130
8,2,F,F,1


In [16]:
vertex_records = []

for road in roads_m[
    ["geometry", "layer", "bridge", "tunnel"]
].itertuples():
    coordinates = list(road.geometry.coords)

    for position, coordinate in enumerate(coordinates):
        x, y = coordinate[:2]

        vertex_records.append({
            "source_row": road.Index,
            "position": position,
            "x": x,
            "y": y,
            "grade": (int(road.layer), road.bridge, road.tunnel),
            "is_endpoint": position in (0, len(coordinates) - 1),
        })

vertices = pd.DataFrame(vertex_records)

# Eyni koordinatda neçə yol və neçə səviyyə tipi var?
coordinate_summary = vertices.groupby(
    ["x", "y"], sort=False
).agg(
    road_count=("source_row", "nunique"),
    grade_count=("grade", "nunique"),
    all_endpoints=("is_endpoint", "all"),
)

mixed_grade = coordinate_summary[
    coordinate_summary["grade_count"] > 1
]

print("Yollardakı ümumi koordinat sayı:", len(vertices))
print("Unikal koordinat sayı:", len(coordinate_summary))
print(
    "Birdən çox yolun paylaşdığı koordinatlar:",
    coordinate_summary["road_count"].gt(1).sum(),
)
print("Fərqli səviyyə tiplərinin görüşdüyü koordinatlar:", len(mixed_grade))
print(
    "Bunlardan bütün yolların uc nöqtəsi olanlar:",
    mixed_grade["all_endpoints"].sum(),
)

display(mixed_grade.head(10))

Yollardakı ümumi koordinat sayı: 155982
Unikal koordinat sayı: 118261
Birdən çox yolun paylaşdığı koordinatlar: 30697
Fərqli səviyyə tiplərinin görüşdüyü koordinatlar: 716
Bunlardan bütün yolların uc nöqtəsi olanlar: 648


,,road_count,grade_count,all_endpoints
x,y,,,
397345.319362,4.476056e+06,2,2,True
397263.769347,4.476134e+06,2,2,True
398042.714289,4.475095e+06,2,2,True
397359.789021,4.476069e+06,2,2,True
397276.233294,4.476149e+06,2,2,True
396437.468870,4.477011e+06,2,2,True
401679.883698,4.469852e+06,2,2,True
401700.574438,4.469859e+06,2,2,True
401452.959194,4.469183e+06,2,2,True


In [17]:
# Daxili nöqtə iştirak edirsə, fərqli səviyyələri ayırırıq.
separate_grade_coords = set(
    mixed_grade.index[~mixed_grade["all_endpoints"]]
)

assumed_transition_coords = set(
    mixed_grade.index[mixed_grade["all_endpoints"]]
)

node_keys = []

for vertex in vertices.itertuples(index=False):
    xy = (vertex.x, vertex.y)

    # Eyni koordinatda fərqli səviyyələr ayrı düyün ala bilər.
    grade_key = vertex.grade if xy in separate_grade_coords else None
    node_keys.append((vertex.x, vertex.y, grade_key))

# Eyni açara malik koordinatlar eyni düyün nömrəsini alır.
vertices["node_id"], _ = pd.factorize(
    pd.Series(node_keys, dtype="object"),
    sort=False,
)

nodes = vertices.drop_duplicates("node_id")[
    ["node_id", "x", "y"]
].copy()

nodes["endpoint_transition_assumed"] = [
    (x, y) in assumed_transition_coords
    for x, y in nodes[["x", "y"]].itertuples(index=False, name=None)
]

print("Qrafın düyün sayı:", len(nodes))
print(
    "Fərziyyə ilə birləşdirilmiş keçid düyünləri:",
    nodes["endpoint_transition_assumed"].sum(),
)

Qrafın düyün sayı: 118329
Fərziyyə ilə birləşdirilmiş keçid düyünləri: 648


In [18]:
import numpy as np

next_points = vertices.groupby(
    "source_row", sort=False
)[["node_id", "x", "y"]].shift(-1)

# Yolun son nöqtəsindən sonra bağlantı yaratmırıq.
has_next = next_points["node_id"].notna()

edges = vertices.loc[
    has_next, ["source_row", "position", "node_id", "x", "y"]
].rename(columns={
    "node_id": "u",
    "x": "x0",
    "y": "y0",
}).copy()

edges["v"] = next_points.loc[has_next, "node_id"].astype("int64")
edges["x1"] = next_points.loc[has_next, "x"]
edges["y1"] = next_points.loc[has_next, "y"]

# İki qonşu koordinat arasındakı məsafə, metrlə.
edges["length_m"] = np.hypot(
    edges["x1"] - edges["x0"],
    edges["y1"] - edges["y0"],
)

# Təkrarlanan qonşu koordinatların sıfır uzunluqlu hissələrini ayırırıq.
zero_length_count = edges["length_m"].eq(0).sum()
edges = edges.loc[edges["length_m"] > 0].copy()

# ID eyni raw fayl daxilində sabitdir.
edges["edge_id"] = (
    "roadrow:" + edges["source_row"].astype(str)
    + ":segment:" + edges["position"].astype(str)
)

attributes = roads_m[[
    "osm_id", "fclass", "layer", "bridge", "tunnel",
    "walk_forward", "walk_reverse",
    "bike_forward", "bike_reverse",
    "drive_forward", "drive_reverse",
]].copy()

edges = edges.join(
    attributes,
    on="source_row",
    validate="many_to_one",
).reset_index(drop=True)

# Hissələrin cəmi orijinal yolun uzunluğuna bərabər olmalıdır.
rebuilt_lengths = edges.groupby("source_row")["length_m"].sum()
length_error = (
    rebuilt_lengths.reindex(roads_m.index, fill_value=0)
    - roads_m["length_m"]
).abs().max()

if length_error > 0.000001:
    raise ValueError(f"Yol uzunluğu uyğun gəlmir: {length_error} metr")

print("Bağlantı hissələrinin sayı:", len(edges))
print("Çıxarılan sıfır uzunluqlu hissələr:", zero_length_count)
print("Maksimum uzunluq fərqi, metr:", length_error)

display(edges[["edge_id", "u", "v", "length_m", "fclass"]].head())

Bağlantı hissələrinin sayı: 130962
Çıxarılan sıfır uzunluqlu hissələr: 0
Maksimum uzunluq fərqi, metr: 2.2737367544323206e-12


,edge_id,u,v,length_m,fclass
0,roadrow:0:segment:0,0,1,13.838229,secondary_link
1,roadrow:0:segment:1,1,2,5.085486,secondary_link
2,roadrow:0:segment:2,2,3,4.747605,secondary_link
3,roadrow:0:segment:3,3,4,5.718992,secondary_link
4,roadrow:0:segment:4,4,5,10.612036,secondary_link


In [19]:
%pip install scipy

   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/36.6 MB ? eta -:--:--
    --------------------------------------- 0.5/36.6 MB 3.2 MB/s eta 0:00:12
   - -------------------------------------- 1.3/36.6 MB 3.1 MB/s eta 0:00:12
   -- ------------------------------------- 2.1/36.6 MB 3.0 MB/s eta 0:00:12
   -- ------------------------------------- 2.6/36.6 MB 3.1 MB/s eta 0:00:12
   --- ------------------------------------ 3.1/36.6 MB 3.0 MB/s eta 0:00:12
   --- ------------------------------------ 3.4/36.6 MB 2.8 MB/s eta 0:00:12
   ---- ----------------------------------- 3.9/36.6 MB 2.7 MB/s eta 0:00:13
   ---- ----------------------------------- 4.5/36.6 MB 2.5 MB/s eta 0:00:13
   ----- ---------------------------------- 4.7/36.6 MB 2.5 MB/s eta 0:00:13
   ----- ---------------------------------- 4.7/36.6 MB 2.5 MB/s eta 0:00:13
   ----- ---------------------------------- 5.0/36.6 MB 2.3 MB/s eta 0:00:14
   ----- ----

In [20]:
from scipy.sparse import csr_matrix

def build_distance_graph(edge_table, node_count, mode):
    if mode not in {"walk", "bike", "drive"}:
        raise ValueError(f"Naməlum rejim: {mode}")

    columns = ["u", "v", "length_m", "edge_id"]

    forward = edge_table.loc[
        edge_table[f"{mode}_forward"], columns
    ].copy()
    forward["direction"] = "forward"

    reverse = edge_table.loc[
        edge_table[f"{mode}_reverse"], columns
    ].rename(columns={"u": "v", "v": "u"}).copy()
    reverse["direction"] = "reverse"

    arcs = pd.concat([forward, reverse], ignore_index=True)

    lengths = arcs["length_m"].to_numpy()
    if not np.isfinite(lengths).all() or (lengths <= 0).any():
        raise ValueError("Bağlantı uzunluqları sonlu və müsbət olmalıdır.")

    # Eyni iki düyün arasında paralel yollar varsa, ən qısasını seçirik.
    # edge_id saxlanılır ki, seçilən yolun mənbəyini itirməyək.
    arcs = (
        arcs.sort_values(
            ["u", "v", "length_m", "edge_id", "direction"]
        )
        .drop_duplicates(["u", "v"])
        .reset_index(drop=True)
    )

    graph = csr_matrix(
        (
            arcs["length_m"].to_numpy(),
            (arcs["u"].to_numpy(), arcs["v"].to_numpy()),
        ),
        shape=(node_count, node_count),
    )

    return graph, arcs


walk_graph_m, walk_arcs = build_distance_graph(edges, len(nodes), "walk")
bike_graph_m, bike_arcs = build_distance_graph(edges, len(nodes), "bike")
drive_graph_m, drive_arcs = build_distance_graph(edges, len(nodes), "drive")

display(pd.DataFrame({
    "mode": ["walk", "bike", "drive"],
    "directed_connections": [
        walk_graph_m.nnz,
        bike_graph_m.nnz,
        drive_graph_m.nnz,
    ],
}))

,mode,directed_connections
0,walk,255614
1,bike,190768
2,drive,185529


In [21]:
from scipy.sparse.csgraph import connected_components

_, walk_component_labels = connected_components(
    walk_graph_m,
    directed=False,
    return_labels=True,
)

# Yalnız piyada bağlantılarında iştirak edən düyünləri sayırıq.
active_walk_nodes = np.unique(
    walk_arcs[["u", "v"]].to_numpy().ravel()
)

component_sizes = (
    pd.Series(walk_component_labels[active_walk_nodes])
    .value_counts()
    .rename_axis("component_id")
    .rename("node_count")
)

main_walk_component = int(component_sizes.index[0])
main_node_count = int(component_sizes.iloc[0])

nodes["walk_active"] = nodes["node_id"].isin(active_walk_nodes)
nodes["walk_component"] = walk_component_labels[
    nodes["node_id"].to_numpy()
]
nodes["in_main_walk_component"] = (
    nodes["walk_active"]
    & nodes["walk_component"].eq(main_walk_component)
)

print("Aktiv piyada düyünləri:", len(active_walk_nodes))
print("Piyada komponentlərinin sayı:", len(component_sizes))
print("Ən böyük komponentdə düyün:", main_node_count)
print(
    "Ən böyük komponentin payı:",
    f"{main_node_count / len(active_walk_nodes):.2%}",
)

display(component_sizes.head(10).to_frame())

Aktiv piyada düyünləri: 115717
Piyada komponentlərinin sayı: 142
Ən böyük komponentdə düyün: 113894
Ən böyük komponentin payı: 98.42%


,node_count
component_id,
0,113894
717,274
2278,123
75,96
912,67
1607,57
1703,43
936,41
693,40


In [22]:
from shapely.geometry import LineString

# Piyada istifadəsinə açıq yol hissələrinin həndəsəsini yaradırıq.
walk_segments = edges.loc[
    edges["walk_forward"] | edges["walk_reverse"]
].copy()

walk_segments["component_id"] = walk_component_labels[
    walk_segments["u"].to_numpy()
]

segment_geometry = [
    LineString([(row.x0, row.y0), (row.x1, row.y1)])
    for row in walk_segments[
        ["x0", "y0", "x1", "y1"]
    ].itertuples(index=False)
]

walk_segments_m = gpd.GeoDataFrame(
    walk_segments[["edge_id", "u", "v", "component_id"]].copy(),
    geometry=segment_geometry,
    crs=METRIC_CRS,
)

# Yalnız ilkin iş sahəsindəki metro çıxışlarına baxırıq.
inside_area = metro_exits_m.geometry.intersects(
    routing_area_m.iloc[0]
)

metro_for_snap = metro_exits_m.loc[
    inside_area, ["station_name", "exit_name", "geometry"]
].copy()
metro_for_snap["exit_row"] = metro_for_snap.index

metro_nearest = gpd.sjoin_nearest(
    metro_for_snap,
    walk_segments_m,
    how="left",
    distance_col="snap_distance_m",
)

# Bərabər məsafəli uyğunluqlarda deterministik seçim.
metro_nearest = (
    metro_nearest.sort_values(
        ["exit_row", "snap_distance_m", "edge_id"]
    )
    .drop_duplicates("exit_row")
    .set_index("exit_row")
)

metro_nearest["on_main_component"] = (
    metro_nearest["component_id"] == main_walk_component
)

report_columns = [
    "station_name", "exit_name", "snap_distance_m",
    "component_id", "on_main_component",
]

print("İş sahəsində metro çıxışı:", len(metro_nearest))

print("\nƏn yaxın yolu əsas komponentdə olmayan çıxışlar:")
display(
    metro_nearest.loc[
        ~metro_nearest["on_main_component"], report_columns
    ]
)

print("\nYola məsafəsi ən böyük olan 10 çıxış:")
display(
    metro_nearest[report_columns]
    .sort_values("snap_distance_m", ascending=False)
    .head(10)
)

İş sahəsində metro çıxışı: 69

Ən yaxın yolu əsas komponentdə olmayan çıxışlar:


,station_name,exit_name,snap_distance_m,component_id,on_main_component
exit_row,,,,,
56,Ulduz,Çıxış 1,2.412761,716,False
57,Ulduz,Çıxış 2,5.590939,716,False



Yola məsafəsi ən böyük olan 10 çıxış:


,station_name,exit_name,snap_distance_m,component_id,on_main_component
exit_row,,,,,
45,28.05.2026 00:00:00,Çıxış 2,19.528423,0,True
36,İnşaatçılar,Çıxış 3,17.449406,0,True
35,İnşaatçılar,Çıxış 2,17.044136,0,True
26,Dərnəgül,Çıxış 4,17.025895,0,True
97,Bakmil,Çıxış 1,14.198519,0,True
37,İnşaatçılar,Çıxış 4,12.263759,0,True
10,Memar Əcəmi 2,Çıxış 2,11.187053,0,True
24,Dərnəgül,Çıxış 2,9.666038,0,True
19,Azadlıq prospekti,Çıxış 1,8.638652,0,True


In [23]:
# Mənbədəki adı ayrıca qoruyuruq.
metro["station_name_source"] = metro["station_name"]

# Yalnız təsdiqlədiyimiz səhv yazılışı düzəldirik.
metro["station_name"] = metro["station_name"].replace({
    "28.05.2026 00:00:00": "28 May",
})

In [26]:
metro_exits_m = metro_exits.to_crs(METRIC_CRS)

In [27]:
metro_exits_m

,station_name,station_en,lat,lon,address,exit_name,geometry
0,Xocəsən,Khojasan,40.421499,49.778066,Dairəvi yol,Çıxış 1,POINT (396338.961 4475257.985)
1,Avtovağzal,Avtovaghzal,40.421760,49.796061,Bakı Sumqayıt yolu,Çıxış 1,POINT (397865.973 4475265.998)
2,Avtovağzal,Avtovaghzal,40.421344,49.795113,Bakı Sumqayıt yolu,Çıxış 2,POINT (397784.919 4475220.918)
3,Avtovağzal,Avtovaghzal,40.420451,49.795884,Bakı Sumqayıt yolu,Çıxış 3,POINT (397848.976 4475120.901)
4,Avtovağzal,Avtovaghzal,40.420312,49.795509,Bakı Sumqayıt yolu,Çıxış 4,POINT (397816.952 4475105.905)
...,...,...,...,...,...,...,...
93,İçərişəhər,Icherisheher,40.365899,49.831529,İstiqlaliyyət küçəsi,Çıxış 1,POINT (400792.947 4469024.943)
94,Cəfər Cabbarlı,Jafar Jabbarly,40.379585,49.848716,Dilarə Əliyeva küçəsi,Çıxış 1,POINT (402271.988 4470524.968)
95,Cəfər Cabbarlı,Jafar Jabbarly,40.379608,49.849222,Dilarə Əliyeva küçəsi,Çıxış 2,POINT (402314.975 4470526.962)
96,Şah İsmayıl Xətai,Shah Ismail Khatai,40.383066,49.871971,Xocalı prospekti,Çıxış 1,POINT (404250.986 4470885.921)


In [28]:
metro_exits = gpd.GeoDataFrame(
    metro.copy(),
    geometry=gpd.points_from_xy(metro["lon"], metro["lat"]),
    crs="EPSG:4326",
)

metro_exits_m = metro_exits.to_crs(METRIC_CRS)

# Yalnız düzəltdiyimiz iki çıxışı yoxlayırıq.
display(
    metro_exits_m.loc[
        metro_exits_m["station_en"].eq("28 May"),
        ["station_name_source", "station_name", "exit_name"],
    ]
)

,station_name_source,station_name,exit_name
44,28.05.2026 00:00:00,28 May,Çıxış 1
45,28.05.2026 00:00:00,28 May,Çıxış 2


In [29]:
inside_area = metro_exits_m.geometry.intersects(
    routing_area_m.iloc[0]
)

metro_for_snap = metro_exits_m.loc[
    inside_area, ["station_name", "exit_name", "geometry"]
].copy()
metro_for_snap["exit_row"] = metro_for_snap.index

metro_nearest = gpd.sjoin_nearest(
    metro_for_snap,
    walk_segments_m,
    how="left",
    distance_col="snap_distance_m",
)

metro_nearest = (
    metro_nearest.sort_values(
        ["exit_row", "snap_distance_m", "edge_id"]
    )
    .drop_duplicates("exit_row")
    .set_index("exit_row")
)

metro_nearest["on_main_component"] = (
    metro_nearest["component_id"] == main_walk_component
)

report_columns = [
    "station_name", "exit_name", "snap_distance_m",
    "component_id", "on_main_component",
]

print("İş sahəsində metro çıxışı:", len(metro_nearest))

print("\nƏn yaxın yolu əsas komponentdə olmayan çıxışlar:")
display(
    metro_nearest.loc[
        ~metro_nearest["on_main_component"], report_columns
    ]
)

print("\nYola məsafəsi ən böyük olan 10 çıxış:")
display(
    metro_nearest[report_columns]
    .sort_values("snap_distance_m", ascending=False)
    .head(10)
)

İş sahəsində metro çıxışı: 69

Ən yaxın yolu əsas komponentdə olmayan çıxışlar:


,station_name,exit_name,snap_distance_m,component_id,on_main_component
exit_row,,,,,
56,Ulduz,Çıxış 1,2.412761,716,False
57,Ulduz,Çıxış 2,5.590939,716,False



Yola məsafəsi ən böyük olan 10 çıxış:


,station_name,exit_name,snap_distance_m,component_id,on_main_component
exit_row,,,,,
45,28 May,Çıxış 2,19.528423,0,True
36,İnşaatçılar,Çıxış 3,17.449406,0,True
35,İnşaatçılar,Çıxış 2,17.044136,0,True
26,Dərnəgül,Çıxış 4,17.025895,0,True
97,Bakmil,Çıxış 1,14.198519,0,True
37,İnşaatçılar,Çıxış 4,12.263759,0,True
10,Memar Əcəmi 2,Çıxış 2,11.187053,0,True
24,Dərnəgül,Çıxış 2,9.666038,0,True
19,Azadlıq prospekti,Çıxış 1,8.638652,0,True


In [30]:
import json

outside_main = ~metro_nearest["on_main_component"]
print("Yoxlanan metro çıxışı:", len(metro_nearest))
print("Ən yaxın yolu əsas komponentdən kənarda:", outside_main.sum())

display(metro_nearest.loc[outside_main, report_columns])

# AYNA terminal məlumatını oxuyuruq.
with ZipFile(ayna_zip) as archive:
    terminal_data = json.loads(
        archive.read("AYNA_Datasets/01_bus_terminals/map.geojson")
    )

terminals_m = gpd.GeoDataFrame.from_features(
    terminal_data["features"],
    crs="EPSG:4326",
).to_crs(METRIC_CRS)

terminal_candidates = terminals_m.loc[
    terminals_m.geometry.intersects(routing_area_m.iloc[0])
].copy()

display(terminal_candidates[["name", "geometry"]])

Yoxlanan metro çıxışı: 69
Ən yaxın yolu əsas komponentdən kənarda: 2


,station_name,exit_name,snap_distance_m,component_id,on_main_component
exit_row,,,,,
56,Ulduz,Çıxış 1,2.412761,716,False
57,Ulduz,Çıxış 2,5.590939,716,False


,name,geometry
31,Bakı Beynalxalq Avtovağzal Kompleksi,POINT Z (398074.354 4474776.45 0)


In [31]:
terminal_point = terminal_candidates.loc[
    terminal_candidates["name"].eq(
        "Bakı Beynalxalq Avtovağzal Kompleksi"
    ),
    ["name", "geometry"],
].copy()

if len(terminal_point) != 1:
    raise ValueError("Avtovağzal üçün tək və dəqiq uyğunluq tapılmadı.")

# Mənbədəki Z=0 dəyərindən yüksəklik məlumatı kimi istifadə etmirik.
terminal_point["geometry"] = gpd.points_from_xy(
    terminal_point.geometry.x,
    terminal_point.geometry.y,
    crs=METRIC_CRS,
)

terminal_nearest = gpd.sjoin_nearest(
    terminal_point,
    walk_segments_m,
    how="left",
    distance_col="snap_distance_m",
)

terminal_nearest = (
    terminal_nearest.sort_values(["snap_distance_m", "edge_id"])
    .head(1)
    .copy()
)

terminal_nearest["on_main_component"] = (
    terminal_nearest["component_id"] == main_walk_component
)

display(terminal_nearest[[
    "name", "snap_distance_m", "edge_id",
    "component_id", "on_main_component",
]])

,name,snap_distance_m,edge_id,component_id,on_main_component
31,Bakı Beynalxalq Avtovağzal Kompleksi,25.299962,roadrow:331533:segment:2,0,True


In [32]:
metro_access = metro_nearest.copy()
metro_access["access_id"] = [
    f"metro_exit:{i}" for i in metro_access.index
]
metro_access["label"] = (
    metro_access["station_name"] + " — " + metro_access["exit_name"]
)

terminal_access = terminal_nearest.copy()
terminal_access["access_id"] = "terminal:baku_international"
terminal_access["label"] = terminal_access["name"]

columns = [
    "access_id", "label", "edge_id", "geometry", "on_main_component"
]

access_points = gpd.GeoDataFrame(
    pd.concat(
        [metro_access[columns], terminal_access[columns]],
        ignore_index=True,
    ),
    geometry="geometry",
    crs=METRIC_CRS,
)

segment_lookup = walk_segments_m.set_index("edge_id")["geometry"]

snap_records = []

for point in access_points.itertuples(index=False):
    line = segment_lookup.loc[point.edge_id]

    # Yol hissəsinin başlanğıcından qoşulma yerinə məsafə.
    offset_m = line.project(point.geometry)
    projected_point = line.interpolate(offset_m)

    snap_records.append({
        "access_id": point.access_id,
        "label": point.label,
        "edge_id": point.edge_id,
        "offset_m": offset_m,
        "fraction": offset_m / line.length,
        "snap_distance_m": point.geometry.distance(projected_point),
        "source_x": point.geometry.x,
        "source_y": point.geometry.y,
        "snap_x": projected_point.x,
        "snap_y": projected_point.y,
        "on_main_component": point.on_main_component,
    })

snap_plan = pd.DataFrame(snap_records)

print("Qoşulması planlaşdırılan nöqtə:", len(snap_plan))
print("Toxunulan fərqli yol hissəsi:", snap_plan["edge_id"].nunique())

display(
    snap_plan[[
        "access_id", "label", "fraction",
        "snap_distance_m", "on_main_component",
    ]]
    .sort_values("snap_distance_m", ascending=False)
    .head(10)
)

Qoşulması planlaşdırılan nöqtə: 70
Toxunulan fərqli yol hissəsi: 70


,access_id,label,fraction,snap_distance_m,on_main_component
69,terminal:baku_international,Bakı Beynalxalq Avtovağzal Kompleksi,0.365428,25.299962,True
45,metro_exit:45,28 May — Çıxış 2,0.485852,19.528423,True
36,metro_exit:36,İnşaatçılar — Çıxış 3,0.140071,17.449406,True
35,metro_exit:35,İnşaatçılar — Çıxış 2,0.047614,17.044136,True
26,metro_exit:26,Dərnəgül — Çıxış 4,0.634798,17.025895,True
68,metro_exit:97,Bakmil — Çıxış 1,0.895365,14.198519,True
37,metro_exit:37,İnşaatçılar — Çıxış 4,0.914292,12.263759,True
10,metro_exit:10,Memar Əcəmi 2 — Çıxış 2,0.981778,11.187053,True
24,metro_exit:24,Dərnəgül — Çıxış 2,0.091924,9.666038,True
19,metro_exit:19,Azadlıq prospekti — Çıxış 1,0.801456,8.638652,True


In [33]:
base_nodes = nodes.assign(node_kind="road")
base_edges = edges.assign(
    parent_edge_id=edges["edge_id"],
    edge_type="road",
)

node_lookup = base_nodes.set_index("node_id", drop=False)
edge_lookup = base_edges.set_index("edge_id", drop=False)

new_nodes = []
replacement_edges = []
access_anchor = {}

next_node_id = len(base_nodes)

# Yalnız hesablama dəqiqliyi üçün: 0.00000001 metr.
EPS_M = 1e-8

ordered_snaps = snap_plan.sort_values(
    ["edge_id", "offset_m", "access_id"]
)

for edge_id, group in ordered_snaps.groupby("edge_id", sort=True):
    parent = edge_lookup.loc[edge_id].to_dict()

    # (Yol boyunca məsafə, düyün ID-si, x, y)
    cuts = [(0.0, int(parent["u"]), parent["x0"], parent["y0"])]

    for point in group.itertuples(index=False):
        offset = point.offset_m

        if offset <= EPS_M:
            anchor = int(parent["u"])

        elif parent["length_m"] - offset <= EPS_M:
            anchor = int(parent["v"])

        elif abs(offset - cuts[-1][0]) <= EPS_M:
            # Eyni yerə düşən girişlər eyni yol düyününü paylaşır.
            anchor = cuts[-1][1]

        else:
            anchor = next_node_id
            next_node_id += 1

            node = node_lookup.loc[int(parent["u"])].to_dict()
            node.update({
                "node_id": anchor,
                "x": point.snap_x,
                "y": point.snap_y,
                "node_kind": "road_snap",
                "endpoint_transition_assumed": False,
            })
            new_nodes.append(node)

            cuts.append((offset, anchor, point.snap_x, point.snap_y))

        access_anchor[point.access_id] = anchor

    cuts.append((
        parent["length_m"],
        int(parent["v"]),
        parent["x1"],
        parent["y1"],
    ))

    # Bölünən hissələr əvvəlki yolun bütün rejim qaydalarını saxlayır.
    for part_number, (start, end) in enumerate(zip(cuts, cuts[1:])):
        child = parent.copy()
        child.update({
            "edge_id": f"{edge_id}:part:{part_number}",
            "u": start[1],
            "v": end[1],
            "x0": start[2],
            "y0": start[3],
            "x1": end[2],
            "y1": end[3],
            "length_m": float(np.hypot(
                end[2] - start[2],
                end[3] - start[3],
            )),
        })
        replacement_edges.append(child)

# Köhnə hissəni çıxarıb onun bölünmüş hissələrini yerləşdiririk.
untouched_edges = base_edges.loc[
    ~base_edges["edge_id"].isin(snap_plan["edge_id"])
]

edges_split = pd.concat(
    [untouched_edges, pd.DataFrame(replacement_edges)],
    ignore_index=True,
)

nodes_split = pd.concat(
    [base_nodes, pd.DataFrame(new_nodes)],
    ignore_index=True,
)

access_anchors = snap_plan.copy()
access_anchors["anchor_node_id"] = (
    access_anchors["access_id"].map(access_anchor).astype("int64")
)

# Bölünmə nəticəsində yol uzunluğu dəyişməməlidir.
split_lengths = edges_split.groupby("parent_edge_id")["length_m"].sum()
split_error = (
    split_lengths.reindex(edge_lookup.index)
    - edge_lookup["length_m"]
).abs().max()

if split_error > 1e-6:
    raise ValueError(f"Bölünmədə uzunluq uyğunsuzluğu: {split_error} metr")

print("Yeni yol düyünləri:", len(new_nodes))
print("Bölünmədən sonra düyün sayı:", len(nodes_split))
print("Bölünmədən sonra yol hissələri:", len(edges_split))
print("Maksimum uzunluq fərqi, metr:", split_error)
print("Qoşulma düyünü müəyyən edilmiş giriş:", len(access_anchors))

Yeni yol düyünləri: 62
Bölünmədən sonra düyün sayı: 118391
Bölünmədən sonra yol hissələri: 131024
Maksimum uzunluq fərqi, metr: 2.842170943040401e-14
Qoşulma düyünü müəyyən edilmiş giriş: 70


In [34]:
split_node_lookup = nodes_split.set_index("node_id", drop=False)

access_nodes = []
access_links = []
access_node_map = {}

for point in access_anchors.itertuples(index=False):
    anchor_id = int(point.anchor_node_id)
    anchor = split_node_lookup.loc[anchor_id]

    distance_m = float(np.hypot(
        point.source_x - anchor["x"],
        point.source_y - anchor["y"],
    ))

    # Giriş artıq yol düyününün üstündədirsə, onu paylaşır.
    if distance_m <= EPS_M:
        access_node_map[point.access_id] = anchor_id
        continue

    access_node_id = len(nodes_split) + len(access_nodes)

    node = anchor.to_dict()
    node.update({
        "node_id": access_node_id,
        "x": point.source_x,
        "y": point.source_y,
        "node_kind": "access",
        "endpoint_transition_assumed": False,
    })
    access_nodes.append(node)
    access_node_map[point.access_id] = access_node_id

    access_links.append({
        "edge_id": f"access_link:{point.access_id}",
        "access_id": point.access_id,
        "snapped_edge_id": point.edge_id,
        "edge_type": "access",
        "source_status": "assumption",
        "fclass": "access_connector",
        "u": access_node_id,
        "v": anchor_id,
        "x0": point.source_x,
        "y0": point.source_y,
        "x1": anchor["x"],
        "y1": anchor["y"],
        "length_m": distance_m,
        "walk_forward": True,
        "walk_reverse": True,
        "bike_forward": False,
        "bike_reverse": False,
        "drive_forward": False,
        "drive_reverse": False,
    })

nodes_access = pd.concat(
    [nodes_split, pd.DataFrame(access_nodes)],
    ignore_index=True,
)

edges_access = pd.concat(
    [edges_split, pd.DataFrame(access_links)],
    ignore_index=True,
)

nodes_access["node_id"] = nodes_access["node_id"].astype("int64")
edges_access = edges_access.astype({"u": "int64", "v": "int64"})

access_table = access_anchors.copy()
access_table["node_id"] = (
    access_table["access_id"].map(access_node_map).astype("int64")
)

# Qrafları bölünmüş yollar və giriş bağlantıları ilə yeniləyirik.
walk_graph_access_m, walk_arcs_access = build_distance_graph(
    edges_access, len(nodes_access), "walk"
)
bike_graph_access_m, bike_arcs_access = build_distance_graph(
    edges_access, len(nodes_access), "bike"
)
drive_graph_access_m, drive_arcs_access = build_distance_graph(
    edges_access, len(nodes_access), "drive"
)

print("Yeni giriş düyünləri:", len(access_nodes))
print("Əlavə edilmiş piyada giriş bağlantıları:", len(access_links))
print("Ümumi düyün sayı:", len(nodes_access))
print("Piyada qrafında istiqamətli bağlantı:", walk_graph_access_m.nnz)

display(access_table[[
    "access_id", "label", "node_id", "anchor_node_id", "snap_distance_m"
]].head())

Yeni giriş düyünləri: 70
Əlavə edilmiş piyada giriş bağlantıları: 70
Ümumi düyün sayı: 118461
Piyada qrafında istiqamətli bağlantı: 255878


,access_id,label,node_id,anchor_node_id,snap_distance_m
0,metro_exit:0,Xocəsən — Çıxış 1,118391,118336,7.149384
1,metro_exit:1,Avtovağzal — Çıxış 1,118392,59848,4.119860
2,metro_exit:2,Avtovağzal — Çıxış 2,118393,118379,2.146493
3,metro_exit:3,Avtovağzal — Çıxış 3,118394,118354,0.746187
4,metro_exit:4,Avtovağzal — Çıxış 4,118395,118355,0.813078


In [35]:
from scipy.sparse.csgraph import dijkstra

terminal_rows = access_table.loc[
    access_table["access_id"].eq("terminal:baku_international")
]

if len(terminal_rows) != 1:
    raise ValueError("Terminalın qraf düyünü dəqiq müəyyən edilməyib.")

terminal_row = terminal_rows.iloc[0]
terminal_node_id = int(terminal_row["node_id"])

# Yalnız Avtovağzal stansiyasının çıxışlarını seçirik.
avtovagzal_ids = [
    f"metro_exit:{i}"
    for i in metro_nearest.index[
        metro_nearest["station_name"].eq("Avtovağzal")
    ]
]

terminal_destinations = access_table.loc[
    access_table["access_id"].isin(avtovagzal_ids)
].copy()

if terminal_destinations.empty:
    raise ValueError("Avtovağzal metro çıxışları tapılmadı.")

# Predecessors sonradan yolun xəritədə göstərilməsi üçün saxlanılır.
terminal_walk_distances_m, terminal_walk_predecessors = dijkstra(
    walk_graph_access_m,
    directed=True,
    indices=terminal_node_id,
    return_predecessors=True,
)

terminal_destinations["network_distance_m"] = terminal_walk_distances_m[
    terminal_destinations["node_id"].to_numpy()
]

# Yalnız müqayisə üçün düz xətt məsafəsi.
terminal_destinations["straight_distance_m"] = np.hypot(
    terminal_destinations["source_x"] - terminal_row["source_x"],
    terminal_destinations["source_y"] - terminal_row["source_y"],
)

terminal_destinations["path_exists"] = np.isfinite(
    terminal_destinations["network_distance_m"]
)

display(
    terminal_destinations[[
        "access_id", "label", "network_distance_m",
        "straight_distance_m", "path_exists",
    ]]
    .sort_values(["network_distance_m", "access_id"])
    .round(2)
)

,access_id,label,network_distance_m,straight_distance_m,path_exists
3,metro_exit:3,Avtovağzal — Çıxış 3,487.35,411.63,True
4,metro_exit:4,Avtovağzal — Çıxış 4,520.19,418.09,True
2,metro_exit:2,Avtovağzal — Çıxış 2,621.27,530.40,True
1,metro_exit:1,Avtovağzal — Çıxış 1,805.00,532.05,True


In [36]:
%pip install folium

  Using cached folium-0.20.0-py2.py3-none-any.whl.metadata (4.2 kB)
  Using cached branca-0.8.2-py3-none-any.whl.metadata (1.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
Using cached folium-0.20.0-py2.py3-none-any.whl (113 kB)
Using cached branca-0.8.2-py3-none-any.whl (26 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)

   ---- ----------------------------------- 1/9 [urllib3]
   ---- ----------------------------------- 1/9 [urllib3]
   ---- ----------------------------------- 1/9 [urllib3]
   ------------- -------------------------- 3/9 [idna]
   ------------- -------------------------- 3/9 [idna]
   ----------------- ---------------------- 4/9 [charset_normalizer]
   ----------------- ---------------------- 4/9 [charset_normalizer]
   ---------------------- ----------------- 5/9 [requests]
   -------------------------- ------------- 

In [37]:
import folium
from shapely.geometry import LineString

# Ən yaxın, yol tapılmış çıxışı seçirik.
reachable = terminal_destinations.loc[
    terminal_destinations["path_exists"]
].sort_values(["network_distance_m", "access_id"])

if reachable.empty:
    raise ValueError("Terminaldan metro çıxışlarına yol tapılmadı.")

destination = reachable.iloc[0]
target_node_id = int(destination["node_id"])

# Dijkstra-nın saxladığı əvvəlki düyünlərdən yolu bərpa edirik.
terminal_route_nodes = [target_node_id]
visited = {target_node_id}

while terminal_route_nodes[-1] != terminal_node_id:
    previous = int(
        terminal_walk_predecessors[terminal_route_nodes[-1]]
    )

    if previous < 0 or previous in visited:
        raise ValueError("Yolun bərpasında uyğunsuzluq var.")

    terminal_route_nodes.append(previous)
    visited.add(previous)

terminal_route_nodes.reverse()

# Seçilmiş bağlantıların uzunluğunu ayrıca yoxlayırıq.
pairs = list(zip(
    terminal_route_nodes[:-1],
    terminal_route_nodes[1:],
))

terminal_route_arcs = (
    walk_arcs_access.set_index(["u", "v"])
    .loc[pairs]
    .reset_index()
)

if not np.isclose(
    terminal_route_arcs["length_m"].sum(),
    destination["network_distance_m"],
    rtol=1e-9,
    atol=1e-6,
):
    raise ValueError("Bərpa edilmiş yolun uzunluğu nəticə ilə uyğun deyil.")

route_nodes_data = (
    nodes_access.set_index("node_id")
    .loc[terminal_route_nodes]
    .copy()
)

route_line_m = LineString(
    route_nodes_data[["x", "y"]].to_numpy()
)

route_line_wgs = gpd.GeoSeries(
    [route_line_m], crs=METRIC_CRS
).to_crs("EPSG:4326").iloc[0]

route_latlon = [(y, x) for x, y in route_line_wgs.coords]

terminal_map = folium.Map(
    location=route_latlon[0],
    zoom_start=17,
    control_scale=True,
)

folium.PolyLine(
    route_latlon,
    color="blue",
    weight=5,
    tooltip=f"{destination['network_distance_m']:.2f} m",
).add_to(terminal_map)

folium.Marker(
    route_latlon[0],
    tooltip="Bakı Beynəlxalq Avtovağzal",
).add_to(terminal_map)

folium.Marker(
    route_latlon[-1],
    tooltip=destination["label"],
).add_to(terminal_map)

# Yol üzərində fərziyyə ilə birləşdirilmiş səviyyə keçidlərini işarələyirik.
assumed_nodes = route_nodes_data.loc[
    route_nodes_data["endpoint_transition_assumed"].eq(True)
]

if not assumed_nodes.empty:
    assumed_points = gpd.GeoSeries(
        gpd.points_from_xy(assumed_nodes["x"], assumed_nodes["y"]),
        crs=METRIC_CRS,
    ).to_crs("EPSG:4326")

    for point in assumed_points:
        folium.CircleMarker(
            [point.y, point.x],
            radius=6,
            color="orange",
            fill=True,
            tooltip="Yoxlanmalı səviyyə keçidi",
        ).add_to(terminal_map)

terminal_map.fit_bounds(route_latlon)

print("Seçilmiş çıxış:", destination["label"])
print("Şəbəkə məsafəsi, metr:", round(destination["network_distance_m"], 2))
print("Yolda fərziyyəli səviyyə keçidi:", len(assumed_nodes))

terminal_map

Seçilmiş çıxış: Avtovağzal — Çıxış 3
Şəbəkə məsafəsi, metr: 487.35
Yolda fərziyyəli səviyyə keçidi: 3


In [39]:
import threading
from functools import partial
from http.server import ThreadingHTTPServer, SimpleHTTPRequestHandler
from IPython.display import IFrame, display
from branca.element import Element

# Xəritə sorğuları səhifənin mənbəyini normal şəkildə bildirsin.
terminal_map.get_root().header.add_child(
    Element(
        '<meta name="referrer" '
        'content="strict-origin-when-cross-origin">'
    ),
    name="citypilot_referrer",
)

map_dir = repo / "outputs"
map_dir.mkdir(parents=True, exist_ok=True)

map_path = map_dir / "terminal_walk_check.html"
terminal_map.save(str(map_path))

# Server yalnız bu kompüterdə işləyir.
# Hüceyrə təkrar işlədiləndə mövcud server istifadə olunur.
if "citypilot_map_server" not in globals():
    handler = partial(
        SimpleHTTPRequestHandler,
        directory=str(map_dir.resolve()),
    )

    citypilot_map_server = ThreadingHTTPServer(
        ("127.0.0.1", 0),
        handler,
    )

    threading.Thread(
        target=citypilot_map_server.serve_forever,
        daemon=True,
    ).start()

port = citypilot_map_server.server_address[1]
map_url = f"http://127.0.0.1:{port}/{map_path.name}"

display(IFrame(src=map_url, width="100%", height=650))

127.0.0.1 - - [17/Oct/2026 15:44:36] "GET /terminal_walk_check.html HTTP/1.1" 200 -


In [41]:
edge_metadata = edges_access.set_index("edge_id")
node_metadata = nodes_access.set_index("node_id")

def road_grade(edge):
    if edge["edge_type"] != "road":
        return None

    return (
        int(edge["layer"]),
        edge["bridge"],
        edge["tunnel"],
    )

review_records = []

for position in range(1, len(terminal_route_nodes) - 1):
    node_id = terminal_route_nodes[position]

    if not node_metadata.loc[node_id, "endpoint_transition_assumed"]:
        continue

    incoming_id = terminal_route_arcs.iloc[position - 1]["edge_id"]
    outgoing_id = terminal_route_arcs.iloc[position]["edge_id"]

    incoming = edge_metadata.loc[incoming_id]
    outgoing = edge_metadata.loc[outgoing_id]

    incoming_grade = road_grade(incoming)
    outgoing_grade = road_grade(outgoing)

    grade_change = (
        incoming_grade != outgoing_grade
        if incoming_grade is not None and outgoing_grade is not None
        else None
    )

    review_records.append({
        "node_id": node_id,
        "incoming_osm_id": incoming.get("osm_id"),
        "outgoing_osm_id": outgoing.get("osm_id"),
        "incoming_grade": incoming_grade,
        "outgoing_grade": outgoing_grade,
        "grade_change": grade_change,
        "review_status": "pending_manual_review",
    })

terminal_grade_review = pd.DataFrame(review_records)

print("Grade formatı: (layer, bridge, tunnel)")
display(terminal_grade_review)

if not terminal_grade_review.empty:
    print(
        "Atributları dəyişən yol keçidləri:",
        terminal_grade_review["grade_change"].eq(True).sum(),
    )

Grade formatı: (layer, bridge, tunnel)


,node_id,incoming_osm_id,outgoing_osm_id,incoming_grade,outgoing_grade,grade_change,review_status
0,114572,1514065893,1514065894,"(-1, F, T)","(0, F, F)",True,pending_manual_review
1,59856,1514065914,415865370,"(0, F, F)","(-1, F, T)",True,pending_manual_review
2,59859,415865375,1426796187,"(-1, F, T)","(0, F, F)",True,pending_manual_review


Atributları dəyişən yol keçidləri: 3


In [42]:
railways_raw = gpd.read_file(
    f"zip://{osm_zip.as_posix()}!gis_osm_railways_free_1.shp"
)

# OSM nəqliyyat nöqtələri: stansiyalar, dayanacaqlar və s.
transport_raw = gpd.read_file(
    f"zip://{osm_zip.as_posix()}!gis_osm_transport_free_1.shp"
)

# Metro relslərini ayrıca saxlayırıq.
metro_tracks_m = railways_raw.loc[
    railways_raw["fclass"].eq("subway")
].copy().to_crs(METRIC_CRS)

# İş sahəsindəki nəqliyyat nöqtələri.
transport_m = transport_raw.to_crs(METRIC_CRS)

transport_area_m = transport_m.loc[
    transport_m.geometry.intersects(routing_area_m.iloc[0])
].copy()

print("OSM metro rels obyektləri:", len(metro_tracks_m))

print("\nİş sahəsində nəqliyyat nöqtələrinin növləri:")
print(transport_area_m["fclass"].value_counts())

# Bunlar namizəddir: adi dəmiryol stansiyaları da daxil ola bilər.
station_candidates = transport_area_m.loc[
    transport_area_m["fclass"].isin(
        ["railway_station", "railway_halt"]
    )
].copy()

print("\nStansiya namizədləri:")
display(
    station_candidates[["osm_id", "fclass", "name"]]
    .sort_values(["name", "osm_id"])
)

OSM metro rels obyektləri: 177

İş sahəsində nəqliyyat nöqtələrinin növləri:
fclass
bus_stop           853
railway_station     32
taxi                10
bus_station          9
railway_halt         3
ferry_terminal       1
Name: count, dtype: int64

Stansiya namizədləri:


,osm_id,fclass,name
840,5996343785,railway_station,20 Sentyabr
78,1930089762,railway_station,20 Yanvar st.
736,5188836356,railway_station,28 May
851,6010348738,railway_station,8 Noyabr
404,4168101319,railway_station,Avtovağzal
29,1148914625,railway_station,Azadlıq prospekti
9,670220144,railway_station,Bakmil
360,3950438053,railway_station,Bakı Dəmiryol Stansiyası
731,5162796046,railway_station,Biləcəri Qəsəbə Stansiyası
202,3080722737,railway_halt,Binəqədi


In [43]:
pilot_station_names = [
    "Dərnəgül", "Azadlıq prospekti", "Nəsimi",
    "Memar Əcəmi", "20 Yanvar", "İnşaatçılar",
    "Elmlər Akademiyası", "Nizami",
    "Xocəsən", "Avtovağzal", "Memar Əcəmi 2", "8 Noyabr",
    "28 May", "Sahil", "Gənclik",
]

station_candidates = station_candidates.copy()
station_candidates["name_clean"] = (
    station_candidates["name"]
    .astype("string")
    .str.strip()
    .replace({"20 Yanvar st.": "20 Yanvar"})
)

match_records = []
match_issues = []

for station_name in pilot_station_names:
    exits = metro_exits_m.loc[
        metro_exits_m["station_name"].eq(station_name)
    ]

    # OSM-də hər iki Memar Əcəmi eyni adla saxlanılıb.
    osm_name = (
        "Memar Əcəmi"
        if station_name == "Memar Əcəmi 2"
        else station_name
    )

    candidates = station_candidates.loc[
        station_candidates["name_clean"].eq(osm_name)
    ]

    if exits.empty or candidates.empty:
        match_issues.append({
            "station_name": station_name,
            "exit_count": len(exits),
            "osm_candidate_count": len(candidates),
        })
        continue

    for candidate in candidates.itertuples():
        exit_distances = exits.geometry.distance(candidate.geometry)

        match_records.append({
            "station_name": station_name,
            "osm_id": candidate.osm_id,
            "osm_name": candidate.name,
            "min_exit_distance_m": exit_distances.min(),
            "mean_exit_distance_m": exit_distances.mean(),
            "nearest_subway_m": metro_tracks_m.geometry.distance(
                candidate.geometry
            ).min(),
            "geometry": candidate.geometry,
        })

station_match_candidates_m = gpd.GeoDataFrame(
    match_records,
    geometry="geometry",
    crs=METRIC_CRS,
)

display(
    station_match_candidates_m.drop(columns="geometry")
    .sort_values(["station_name", "mean_exit_distance_m", "osm_id"])
    .round(1)
)

print("Çatışmayan uyğunluqlar:", match_issues)

,station_name,osm_id,osm_name,min_exit_distance_m,mean_exit_distance_m,nearest_subway_m
5,20 Yanvar,1930089762,20 Yanvar st.,63.1,98.0,0.0
14,28 May,5188836356,28 May,24.2,188.3,3.4
13,8 Noyabr,6010348738,8 Noyabr,28.1,90.6,5.9
10,Avtovağzal,4168101319,Avtovağzal,20.2,90.6,6.9
1,Azadlıq prospekti,1148914625,Azadlıq prospekti,7.9,112.6,0.0
0,Dərnəgül,1924834867,Dərnəgül,93.6,99.7,0.0
7,Elmlər Akademiyası,670217100,Elmlər Akademiyası,5.2,234.3,0.0
16,Gənclik,11622641264,Gənclik,2.3,98.1,8.7
3,Memar Əcəmi,4161594118,Memar Əcəmi,86.4,131.8,0.0
4,Memar Əcəmi,7279336593,Memar Əcəmi,200.8,243.3,3.9


Çatışmayan uyğunluqlar: []


In [44]:
memar_candidates = station_match_candidates_m.loc[
    station_match_candidates_m["station_name"].isin(
        ["Memar Əcəmi", "Memar Əcəmi 2"]
    )
]

display(
    memar_candidates.drop(columns="geometry")
    .sort_values(["station_name", "mean_exit_distance_m", "osm_id"])
    .round(1)
)

,station_name,osm_id,osm_name,min_exit_distance_m,mean_exit_distance_m,nearest_subway_m
3,Memar Əcəmi,4161594118,Memar Əcəmi,86.4,131.8,0.0
4,Memar Əcəmi,7279336593,Memar Əcəmi,200.8,243.3,3.9
12,Memar Əcəmi 2,7279336593,Memar Əcəmi,44.9,77.3,3.9
11,Memar Əcəmi 2,4161594118,Memar Əcəmi,21.1,97.4,0.0


In [45]:
memar_choices = {
    "Memar Əcəmi": "4161594118",
    "Memar Əcəmi 2": "7279336593",
}

selected_stations = []

for station_name, group in station_match_candidates_m.groupby(
    "station_name", sort=False
):
    if station_name in memar_choices:
        selected = group.loc[
            group["osm_id"].astype(str).eq(memar_choices[station_name])
        ].copy()
        basis = "exit_group_distance_and_one_to_one_review"
    else:
        selected = group.copy()
        basis = "name_match"

    if len(selected) != 1:
        raise ValueError(
            f"{station_name}: tək uyğunluq müəyyən edilməyib."
        )

    selected["selection_basis"] = basis
    selected["review_status"] = "pending_topology_review"
    selected_stations.append(selected)

stations_m = gpd.GeoDataFrame(
    pd.concat(selected_stations, ignore_index=True),
    geometry="geometry",
    crs=METRIC_CRS,
)

if set(stations_m["station_name"]) != set(pilot_station_names):
    raise ValueError("Pilot stansiya siyahısı tam uyğun gəlmir.")

if stations_m["osm_id"].astype(str).duplicated().any():
    raise ValueError("Eyni OSM nöqtəsi iki stansiyaya təyin edilib.")

stations_m["station_id"] = (
    "metro_station_osm:" + stations_m["osm_id"].astype(str)
)

print("Seçilmiş pilot stansiya sayı:", len(stations_m))

display(
    stations_m.loc[
        stations_m["station_name"].isin(memar_choices),
        ["station_name", "osm_id", "review_status"],
    ]
)

print("\nMetro relslərindəki name dəyərləri:")
print(metro_tracks_m["name"].value_counts(dropna=False).to_string())

Seçilmiş pilot stansiya sayı: 15


,station_name,osm_id,review_status
3,Memar Əcəmi,4161594118,pending_topology_review
10,Memar Əcəmi 2,7279336593,pending_topology_review



Metro relslərindəki name dəyərləri:
name
NaN                                           141
Bakı Metropoliteni - Bənövşəyi xətt            12
Bakı Metropoliteni - Qırmızı xətt               7
Bakı Metropoliteni - Qırmızı xətt (Bakmil)      6
Bakı Metropoliteni                              5
Bakı metropoliteni - Yaşıl xətt                 2
Bakı Metropoliteni - Yaşıl xətt                 2
Bakı metropoliteni (C. Cabbarlı - Xətai)        2


In [46]:
metro_sequences = {
    "green": [
        "Dərnəgül", "Azadlıq prospekti", "Nəsimi",
        "Memar Əcəmi", "20 Yanvar", "İnşaatçılar",
        "Elmlər Akademiyası", "Nizami", "28 May",
    ],
    "purple": [
        "Xocəsən", "Avtovağzal", "Memar Əcəmi 2", "8 Noyabr",
    ],
    "red": ["Sahil", "28 May", "Gənclik"],
}

metro_sources = {
    "green": "https://metro.gov.az/about-lines/3",
    "purple": "https://metro.gov.az/about-lines/7",
    "red": "https://metro.gov.az/about-lines/2",
    "interchange": "https://metro.gov.az/about-lines",
}

station_lookup = stations_m.set_index("station_name")
link_records = []

def add_metro_link(line_id, start, end, kind):
    start_station = station_lookup.loc[start]
    end_station = station_lookup.loc[end]

    is_closure_link = {start, end} == {"Nizami", "28 May"}

    link_records.append({
        "link_id": (
            f"metro_link:{line_id}:"
            f"{start_station['osm_id']}:{end_station['osm_id']}"
        ),
        "line_id": line_id,
        "link_kind": kind,
        "from_name": start,
        "to_name": end,
        "from_station_id": start_station["station_id"],
        "to_station_id": end_station["station_id"],
        "bidirectional": True,
        "normal_enabled": True,
        "disrupted_enabled": not is_closure_link,
        "is_closure_link": is_closure_link,
        "source_url": metro_sources[line_id],
        "geometry_status": "pending_osm_geometry_review",
    })

for line_id, sequence in metro_sequences.items():
    for start, end in zip(sequence, sequence[1:]):
        add_metro_link(line_id, start, end, "ride")

add_metro_link(
    "interchange", "Memar Əcəmi", "Memar Əcəmi 2", "interchange"
)

metro_links = pd.DataFrame(link_records)

if metro_links["is_closure_link"].sum() != 1:
    raise ValueError("Bağlanacaq metro bağlantısı tək olmalıdır.")

print("Metro səfər bağlantıları:", metro_links["link_kind"].eq("ride").sum())
print("Xətlərarası keçid:", metro_links["link_kind"].eq("interchange").sum())

display(metro_links[[
    "line_id", "link_kind", "from_name", "to_name",
    "normal_enabled", "disrupted_enabled",
]])

Metro səfər bağlantıları: 13
Xətlərarası keçid: 1


,line_id,link_kind,from_name,to_name,normal_enabled,disrupted_enabled
0,green,ride,Dərnəgül,Azadlıq prospekti,True,True
1,green,ride,Azadlıq prospekti,Nəsimi,True,True
2,green,ride,Nəsimi,Memar Əcəmi,True,True
3,green,ride,Memar Əcəmi,20 Yanvar,True,True
4,green,ride,20 Yanvar,İnşaatçılar,True,True
5,green,ride,İnşaatçılar,Elmlər Akademiyası,True,True
6,green,ride,Elmlər Akademiyası,Nizami,True,True
7,green,ride,Nizami,28 May,True,False
8,purple,ride,Xocəsən,Avtovağzal,True,True
9,purple,ride,Avtovağzal,Memar Əcəmi 2,True,True


In [47]:
metro_parts = metro_tracks_m.copy()
metro_parts["source_track_row"] = metro_parts.index

metro_parts = metro_parts.explode(
    index_parts=False,
    ignore_index=True,
)

bad_geometry = (
    metro_parts.geometry.isna()
    | metro_parts.geometry.is_empty
    | ~metro_parts.geometry.is_valid
    | metro_parts.geom_type.ne("LineString")
)

if bad_geometry.any():
    raise ValueError("Metro relslərində yoxlanmalı həndəsə var.")

if metro_parts[["layer", "bridge", "tunnel"]].isna().any().any():
    raise ValueError("Metro relslərində səviyyə məlumatı çatışmır.")

metro_vertex_records = []

for track in metro_parts[
    ["geometry", "layer", "bridge", "tunnel"]
].itertuples():
    coordinates = list(track.geometry.coords)

    for position, coordinate in enumerate(coordinates):
        x, y = coordinate[:2]

        metro_vertex_records.append({
            "track_part": track.Index,
            "position": position,
            "x": x,
            "y": y,
            "grade": (int(track.layer), track.bridge, track.tunnel),
            "is_endpoint": position in (0, len(coordinates) - 1),
        })

metro_vertices = pd.DataFrame(metro_vertex_records)

metro_coordinate_summary = metro_vertices.groupby(
    ["x", "y"], sort=False
).agg(
    track_count=("track_part", "nunique"),
    grade_count=("grade", "nunique"),
    all_endpoints=("is_endpoint", "all"),
)

metro_mixed_grade = metro_coordinate_summary.loc[
    metro_coordinate_summary["grade_count"] > 1
]

print("Metro xətt hissələri:", len(metro_parts))
print("Ümumi rels koordinatları:", len(metro_vertices))
print("Unikal koordinatlar:", len(metro_coordinate_summary))
print("Fərqli səviyyələrin görüşdüyü koordinatlar:", len(metro_mixed_grade))

display(metro_mixed_grade.head(10))

Metro xətt hissələri: 177
Ümumi rels koordinatları: 1349
Unikal koordinatlar: 1152
Fərqli səviyyələrin görüşdüyü koordinatlar: 12


,,track_count,grade_count,all_endpoints
x,y,,,
401847.913489,4.475679e+06,2,2,True
404985.578081,4.473629e+06,2,2,True
404957.794298,4.473641e+06,3,2,True
402233.318048,4.470660e+06,3,2,True
405947.543513,4.474406e+06,2,2,True
404990.281013,4.473710e+06,2,2,True
403039.522354,4.476223e+06,2,2,True
396885.878005,4.475908e+06,2,2,True
402239.604064,4.470662e+06,2,2,False


In [48]:
# Fərqli səviyyələr daxili nöqtədə görüşürsə, ayrı saxlayırıq.
metro_separate_coords = set(
    metro_mixed_grade.index[~metro_mixed_grade["all_endpoints"]]
)

metro_assumed_coords = set(
    metro_mixed_grade.index[metro_mixed_grade["all_endpoints"]]
)

metro_node_keys = [
    (
        point.x,
        point.y,
        point.grade
        if (point.x, point.y) in metro_separate_coords
        else None,
    )
    for point in metro_vertices.itertuples(index=False)
]

metro_vertices["node_id"], _ = pd.factorize(
    pd.Series(metro_node_keys, dtype="object"),
    sort=False,
)

metro_nodes = metro_vertices.drop_duplicates("node_id")[
    ["node_id", "x", "y"]
].copy()

metro_nodes["endpoint_transition_assumed"] = [
    (x, y) in metro_assumed_coords
    for x, y in metro_nodes[["x", "y"]].itertuples(
        index=False, name=None
    )
]

# Hər rels hissəsində qonşu koordinatları birləşdiririk.
metro_next = metro_vertices.groupby(
    "track_part", sort=False
)[["node_id", "x", "y"]].shift(-1)

has_next = metro_next["node_id"].notna()

metro_edges = metro_vertices.loc[
    has_next, ["track_part", "position", "node_id", "x", "y"]
].rename(columns={
    "node_id": "u",
    "x": "x0",
    "y": "y0",
}).copy()

metro_edges["v"] = metro_next.loc[has_next, "node_id"].astype("int64")
metro_edges["x1"] = metro_next.loc[has_next, "x"]
metro_edges["y1"] = metro_next.loc[has_next, "y"]

metro_edges["length_m"] = np.hypot(
    metro_edges["x1"] - metro_edges["x0"],
    metro_edges["y1"] - metro_edges["y0"],
)

metro_zero_count = metro_edges["length_m"].eq(0).sum()
metro_edges = metro_edges.loc[metro_edges["length_m"] > 0].copy()

metro_edges["edge_id"] = (
    "metro_track:" + metro_edges["track_part"].astype(str)
    + ":segment:" + metro_edges["position"].astype(str)
)

metro_edges = metro_edges.join(
    metro_parts[[
        "osm_id", "source_track_row", "name", "layer", "bridge", "tunnel"
    ]],
    on="track_part",
    validate="many_to_one",
).reset_index(drop=True)

# Həndəsə axtarışı üçün hər iki istiqaməti açırıq.
metro_forward = metro_edges[
    ["u", "v", "length_m", "edge_id"]
].assign(direction="forward")

metro_reverse = metro_edges[
    ["u", "v", "length_m", "edge_id"]
].rename(columns={"u": "v", "v": "u"}).assign(direction="reverse")

metro_arcs = (
    pd.concat([metro_forward, metro_reverse], ignore_index=True)
    .sort_values(["u", "v", "length_m", "edge_id", "direction"])
    .drop_duplicates(["u", "v"])
    .reset_index(drop=True)
)

metro_track_graph_m = csr_matrix(
    (
        metro_arcs["length_m"].to_numpy(),
        (metro_arcs["u"].to_numpy(), metro_arcs["v"].to_numpy()),
    ),
    shape=(len(metro_nodes), len(metro_nodes)),
)

metro_component_count, metro_component_labels = connected_components(
    metro_track_graph_m,
    directed=False,
    return_labels=True,
)

print("Rels düyünləri:", len(metro_nodes))
print("Rels bağlantı hissələri:", len(metro_edges))
print("Çıxarılan sıfır uzunluqlu hissələr:", metro_zero_count)
print("Rels komponentləri:", metro_component_count)
print("Fərziyyəli uc-nöqtə keçidləri:", len(metro_assumed_coords))

display(
    pd.Series(metro_component_labels)
    .value_counts()
    .rename_axis("component_id")
    .rename("node_count")
    .head(10)
    .to_frame()
)

Rels düyünləri: 1154
Rels bağlantı hissələri: 1172
Çıxarılan sıfır uzunluqlu hissələr: 0
Rels komponentləri: 5
Fərziyyəli uc-nöqtə keçidləri: 10


,node_count
component_id,
0,679
1,296
3,155
4,16
2,8


In [49]:
metro_segment_geometry = [
    LineString([(edge.x0, edge.y0), (edge.x1, edge.y1)])
    for edge in metro_edges[
        ["x0", "y0", "x1", "y1"]
    ].itertuples(index=False)
]

metro_segments_m = gpd.GeoDataFrame(
    metro_edges[[
        "edge_id", "u", "v", "track_part", "osm_id",
        "name", "layer", "bridge", "tunnel",
    ]].copy(),
    geometry=metro_segment_geometry,
    crs=METRIC_CRS,
)

metro_segments_m["track_component"] = metro_component_labels[
    metro_segments_m["u"].to_numpy()
]

station_snap_candidates = gpd.sjoin_nearest(
    stations_m[["station_id", "station_name", "geometry"]],
    metro_segments_m,
    how="left",
    distance_col="track_distance_m",
)

# Bərabər məsafədə fərqli komponentlər varsa, qeyri-müəyyənliyi saxlayırıq.
component_options = station_snap_candidates.groupby(
    "station_id"
)["track_component"].nunique()

station_snap_candidates["nearest_component_count"] = (
    station_snap_candidates["station_id"].map(component_options)
)

station_track_matches = (
    station_snap_candidates
    .sort_values(["station_id", "track_distance_m", "edge_id"])
    .drop_duplicates("station_id")
    .reset_index(drop=True)
)

display(
    station_track_matches[[
        "station_name", "track_distance_m",
        "track_component", "nearest_component_count",
    ]]
    .sort_values(["track_component", "station_name"])
    .round(2)
)

print("\nKomponentlər üzrə stansiyalar:")
print(
    station_track_matches.groupby("track_component")["station_name"]
    .agg(list)
    .to_string()
)

,station_name,track_distance_m,track_component,nearest_component_count
5,20 Yanvar,0.00,0,1
8,28 May,3.41,0,1
0,Azadlıq prospekti,0.00,0,1
4,Dərnəgül,0.00,0,1
11,Elmlər Akademiyası,0.00,0,1
1,Gənclik,8.73,0,1
6,Memar Əcəmi,0.00,0,1
10,Nizami,0.00,0,1
13,Nəsimi,0.00,0,1
2,Sahil,1.63,0,1



Komponentlər üzrə stansiyalar:
track_component
0    [Azadlıq prospekti, Gənclik, Sahil, Dərnəgül, ...
1       [Xocəsən, Avtovağzal, 8 Noyabr, Memar Əcəmi 2]


In [50]:
metro_segment_lookup = metro_segments_m.set_index("edge_id")["geometry"]

station_snap_records = []

for station in station_track_matches.itertuples(index=False):
    line = metro_segment_lookup.loc[station.edge_id]

    offset_m = line.project(station.geometry)
    projected = line.interpolate(offset_m)

    station_snap_records.append({
        "station_id": station.station_id,
        "station_name": station.station_name,
        "edge_id": station.edge_id,
        "offset_m": offset_m,
        "fraction": offset_m / line.length,
        "snap_x": projected.x,
        "snap_y": projected.y,
        "track_distance_m": station.geometry.distance(projected),
        "track_component": station.track_component,
    })

metro_station_snap_plan = pd.DataFrame(station_snap_records)

print("Relsə yerləşdiriləcək stansiya:", len(metro_station_snap_plan))
print(
    "Toxunulan fərqli rels hissəsi:",
    metro_station_snap_plan["edge_id"].nunique(),
)

display(
    metro_station_snap_plan[[
        "station_name", "fraction",
        "track_distance_m", "track_component",
    ]]
    .sort_values(["track_component", "station_name"])
    .round(3)
)

Relsə yerləşdiriləcək stansiya: 15
Toxunulan fərqli rels hissəsi: 15


,station_name,fraction,track_distance_m,track_component
5,20 Yanvar,1.000,0.000,0
8,28 May,0.121,3.406,0
0,Azadlıq prospekti,0.000,0.000,0
4,Dərnəgül,0.000,0.000,0
11,Elmlər Akademiyası,1.000,0.000,0
1,Gənclik,0.035,8.735,0
6,Memar Əcəmi,1.000,0.000,0
10,Nizami,1.000,0.000,0
13,Nəsimi,1.000,0.000,0
2,Sahil,0.998,1.627,0


In [51]:
# Bu variant hər rels hissəsində bir stansiya olması üçündür.
if not metro_station_snap_plan["edge_id"].is_unique:
    raise ValueError("Eyni rels hissəsinə bir neçə stansiya düşüb.")

metro_base_edges = metro_edges.assign(
    parent_edge_id=metro_edges["edge_id"]
)
metro_edge_lookup = metro_base_edges.set_index("edge_id", drop=False)

metro_new_nodes = []
metro_replacement_edges = []
metro_replaced_ids = set()
station_track_node_map = {}

for station in metro_station_snap_plan.sort_values(
    "station_id"
).itertuples(index=False):
    parent = metro_edge_lookup.loc[station.edge_id].to_dict()

    if station.offset_m <= EPS_M:
        anchor = int(parent["u"])

    elif parent["length_m"] - station.offset_m <= EPS_M:
        anchor = int(parent["v"])

    else:
        anchor = len(metro_nodes) + len(metro_new_nodes)

        metro_new_nodes.append({
            "node_id": anchor,
            "x": station.snap_x,
            "y": station.snap_y,
            "endpoint_transition_assumed": False,
        })

        cuts = [
            (int(parent["u"]), parent["x0"], parent["y0"]),
            (anchor, station.snap_x, station.snap_y),
            (int(parent["v"]), parent["x1"], parent["y1"]),
        ]

        for part_number, (start, end) in enumerate(zip(cuts, cuts[1:])):
            child = parent.copy()
            child.update({
                "edge_id": f"{station.edge_id}:part:{part_number}",
                "u": start[0],
                "v": end[0],
                "x0": start[1],
                "y0": start[2],
                "x1": end[1],
                "y1": end[2],
                "length_m": float(np.hypot(
                    end[1] - start[1],
                    end[2] - start[2],
                )),
            })
            metro_replacement_edges.append(child)

        metro_replaced_ids.add(station.edge_id)

    station_track_node_map[station.station_id] = anchor

metro_nodes_split = pd.concat(
    [metro_nodes, pd.DataFrame(metro_new_nodes)],
    ignore_index=True,
)

metro_edges_split = pd.concat([
    metro_base_edges.loc[
        ~metro_base_edges["edge_id"].isin(metro_replaced_ids)
    ],
    pd.DataFrame(metro_replacement_edges),
], ignore_index=True)

station_track_nodes = metro_station_snap_plan.copy()
station_track_nodes["track_node_id"] = (
    station_track_nodes["station_id"]
    .map(station_track_node_map)
    .astype("int64")
)

# Bölünmə rels uzunluğunu dəyişməməlidir.
rebuilt = metro_edges_split.groupby("parent_edge_id")["length_m"].sum()
metro_split_error = (
    rebuilt.reindex(metro_edge_lookup.index)
    - metro_edge_lookup["length_m"]
).abs().max()

if metro_split_error > 1e-6:
    raise ValueError("Rels bölünməsində uzunluq uyğunsuzluğu var.")

print("Yeni rels düyünləri:", len(metro_new_nodes))
print("Ümumi rels düyünləri:", len(metro_nodes_split))
print("Rels hissələrinin sayı:", len(metro_edges_split))
print("Maksimum uzunluq fərqi, metr:", metro_split_error)
print("Rels düyünü təyin edilmiş stansiya:", len(station_track_nodes))

Yeni rels düyünləri: 7
Ümumi rels düyünləri: 1161
Rels hissələrinin sayı: 1179
Maksimum uzunluq fərqi, metr: 1.4210854715202004e-14
Rels düyünü təyin edilmiş stansiya: 15


In [52]:
columns = ["u", "v", "length_m", "edge_id"]

forward = metro_edges_split[columns].assign(direction="forward")
reverse = (
    metro_edges_split[columns]
    .rename(columns={"u": "v", "v": "u"})
    .assign(direction="reverse")
)

metro_arcs_split = (
    pd.concat([forward, reverse], ignore_index=True)
    .sort_values(["u", "v", "length_m", "edge_id", "direction"])
    .drop_duplicates(["u", "v"])
    .reset_index(drop=True)
)

if (
    not np.isfinite(metro_arcs_split["length_m"]).all()
    or metro_arcs_split["length_m"].le(0).any()
):
    raise ValueError("Rels bağlantılarının uzunluğunda problem var.")

metro_track_graph_split_m = csr_matrix(
    (
        metro_arcs_split["length_m"].to_numpy(),
        (
            metro_arcs_split["u"].to_numpy(dtype="int64"),
            metro_arcs_split["v"].to_numpy(dtype="int64"),
        ),
    ),
    shape=(len(metro_nodes_split), len(metro_nodes_split)),
)

new_component_count, _ = connected_components(
    metro_track_graph_split_m,
    directed=False,
    return_labels=True,
)

if new_component_count != metro_component_count:
    raise ValueError("Bölünmə rels komponentlərinin sayını dəyişib.")

# Bütün 15 stansiyadan məsafələri bir çağırışda hesablayırıq.
metro_station_distances_m, metro_station_predecessors = dijkstra(
    metro_track_graph_split_m,
    directed=True,
    indices=station_track_nodes["track_node_id"].to_numpy(dtype="int64"),
    return_predecessors=True,
)

station_matrix_row = {
    station_id: row
    for row, station_id in enumerate(station_track_nodes["station_id"])
}

metro_ride_links = metro_links.loc[
    metro_links["link_kind"].eq("ride")
].copy()

source_rows = (
    metro_ride_links["from_station_id"]
    .map(station_matrix_row)
    .to_numpy(dtype="int64")
)

target_nodes = (
    metro_ride_links["to_station_id"]
    .map(station_track_node_map)
    .to_numpy(dtype="int64")
)

metro_ride_links["track_length_m"] = metro_station_distances_m[
    source_rows, target_nodes
]

metro_ride_links["geometry_path_exists"] = np.isfinite(
    metro_ride_links["track_length_m"]
)

print("Rels komponentləri qorunub:", new_component_count)
print("Yol tapılan bağlantı:", metro_ride_links["geometry_path_exists"].sum())

display(
    metro_ride_links[[
        "line_id", "from_name", "to_name",
        "track_length_m", "geometry_path_exists",
    ]].round(2)
)

Rels komponentləri qorunub: 5
Yol tapılan bağlantı: 13


,line_id,from_name,to_name,track_length_m,geometry_path_exists
0,green,Dərnəgül,Azadlıq prospekti,1601.38,True
1,green,Azadlıq prospekti,Nəsimi,1420.82,True
2,green,Nəsimi,Memar Əcəmi,2025.13,True
3,green,Memar Əcəmi,20 Yanvar,912.90,True
4,green,20 Yanvar,İnşaatçılar,1963.76,True
5,green,İnşaatçılar,Elmlər Akademiyası,2096.24,True
6,green,Elmlər Akademiyası,Nizami,1326.66,True
7,green,Nizami,28 May,1778.89,True
8,purple,Xocəsən,Avtovağzal,2338.27,True
9,purple,Avtovağzal,Memar Əcəmi 2,6135.01,True


In [53]:
metro_node_lookup = metro_nodes_split.set_index("node_id")

if station_track_nodes["track_node_id"].duplicated().any():
    raise ValueError("İki stansiyaya eyni rels düyünü təyin edilib.")

station_at_node = station_track_nodes.set_index(
    "track_node_id"
)["station_name"].to_dict()

metro_link_paths = {}
geometry_records = []

for link in metro_ride_links.itertuples(index=False):
    if not link.geometry_path_exists:
        continue

    source = station_track_node_map[link.from_station_id]
    target = station_track_node_map[link.to_station_id]
    matrix_row = station_matrix_row[link.from_station_id]

    path = [target]
    visited = {target}

    while path[-1] != source:
        previous = int(
            metro_station_predecessors[matrix_row, path[-1]]
        )

        if previous < 0 or previous in visited:
            raise ValueError(f"Yol bərpasında problem: {link.link_id}")

        path.append(previous)
        visited.add(previous)

    path.reverse()
    metro_link_paths[link.link_id] = path

    coordinates = metro_node_lookup.loc[path, ["x", "y"]].to_numpy()
    geometry = LineString(coordinates)

    direct_m = float(np.linalg.norm(
        coordinates[-1] - coordinates[0]
    ))

    if direct_m <= EPS_M:
        raise ValueError("Fərqli stansiyalar eyni qoşulma yerinə düşüb.")

    intermediate = list(dict.fromkeys(
        station_at_node[node]
        for node in path[1:-1]
        if node in station_at_node
    ))

    geometry_records.append({
        "link_id": link.link_id,
        "line_id": link.line_id,
        "from_name": link.from_name,
        "to_name": link.to_name,
        "track_length_m": link.track_length_m,
        "direct_distance_m": direct_m,
        "detour_ratio": link.track_length_m / direct_m,
        "other_station_nodes_on_path": ", ".join(intermediate),
        "geometry": geometry,
        "review_status": "pending_geometry_review",
    })

metro_link_geometries_m = gpd.GeoDataFrame(
    geometry_records,
    geometry="geometry",
    crs=METRIC_CRS,
)

display(
    metro_link_geometries_m[[
        "from_name", "to_name", "track_length_m",
        "direct_distance_m", "detour_ratio",
        "other_station_nodes_on_path",
    ]]
    .sort_values("detour_ratio", ascending=False)
    .round(2)
)

,from_name,to_name,track_length_m,direct_distance_m,detour_ratio,other_station_nodes_on_path
9,Avtovağzal,Memar Əcəmi 2,6135.01,1912.35,3.21,
11,Sahil,28 May,3092.78,966.19,3.20,
12,28 May,Gənclik,6753.89,2239.93,3.02,Sahil
8,Xocəsən,Avtovağzal,2338.27,1363.99,1.71,
10,Memar Əcəmi 2,8 Noyabr,1445.98,1170.94,1.23,
4,20 Yanvar,İnşaatçılar,1963.76,1730.73,1.13,
7,Nizami,28 May,1778.89,1578.30,1.13,
5,İnşaatçılar,Elmlər Akademiyası,2096.24,1907.09,1.10,
2,Nəsimi,Memar Əcəmi,2025.13,1870.51,1.08,
3,Memar Əcəmi,20 Yanvar,912.90,898.84,1.02,


In [54]:
METRO_CANDIDATE_RADIUS_M = 30.0

station_component_lookup = station_track_matches.set_index(
    "station_id"
)["track_component"]

metro_candidate_records = []

for station in stations_m.itertuples(index=False):
    component = int(station_component_lookup.loc[station.station_id])

    distances = metro_segments_m.geometry.distance(station.geometry)

    nearby = metro_segments_m.loc[
        distances.le(METRO_CANDIDATE_RADIUS_M)
        & metro_segments_m["track_component"].eq(component)
    ]

    if nearby.empty:
        raise ValueError(f"Rels namizədi yoxdur: {station.station_name}")

    for segment in nearby.itertuples():
        line = segment.geometry
        offset = float(line.project(station.geometry))
        projected = line.interpolate(offset)

        if offset <= EPS_M:
            anchor_key = f"node:{segment.u}"
        elif line.length - offset <= EPS_M:
            anchor_key = f"node:{segment.v}"
        else:
            anchor_key = f"edge:{segment.edge_id}"

        metro_candidate_records.append({
            "station_id": station.station_id,
            "station_name": station.station_name,
            "candidate_id": f"{station.station_id}|{anchor_key}",
            "anchor_key": anchor_key,
            "edge_id": segment.edge_id,
            "u": int(segment.u),
            "v": int(segment.v),
            "offset_m": offset,
            "segment_length_m": float(line.length),
            "snap_x": projected.x,
            "snap_y": projected.y,
            "point_distance_m": float(
                station.geometry.distance(projected)
            ),
            "track_component": component,
        })

metro_anchor_candidates = (
    pd.DataFrame(metro_candidate_records)
    .sort_values(["station_id", "point_distance_m", "edge_id"])
    .drop_duplicates(["station_id", "anchor_key"])
    .reset_index(drop=True)
)

display(
    metro_anchor_candidates.groupby("station_name").agg(
        candidate_count=("candidate_id", "nunique"),
        nearest_m=("point_distance_m", "min"),
        farthest_m=("point_distance_m", "max"),
    ).round(2)
)

,candidate_count,nearest_m,farthest_m
station_name,,,
20 Yanvar,1,0.00,0.00
28 May,8,3.41,15.73
8 Noyabr,4,5.91,27.13
Avtovağzal,5,6.94,27.93
Azadlıq prospekti,2,0.00,25.55
Dərnəgül,1,0.00,0.00
Elmlər Akademiyası,2,0.00,22.62
Gənclik,2,8.73,12.83
Memar Əcəmi,1,0.00,0.00


In [55]:
# Namizədlərin aid olduğu mövcud rels ucları.
endpoint_nodes = np.unique(
    metro_anchor_candidates[["u", "v"]].to_numpy(dtype="int64").ravel()
)

endpoint_distances_m, endpoint_predecessors = dijkstra(
    metro_track_graph_m,
    directed=True,
    indices=endpoint_nodes,
    return_predecessors=True,
)

endpoint_matrix_row = {
    int(node): row for row, node in enumerate(endpoint_nodes)
}

def anchor_endpoints(candidate):
    # Mövcud düyünün üstünə düşən namizəd.
    if candidate["anchor_key"].startswith("node:"):
        node = int(candidate["anchor_key"].split(":")[1])
        return [(node, 0.0)]

    return [
        (int(candidate["u"]), float(candidate["offset_m"])),
        (
            int(candidate["v"]),
            max(
                0.0,
                candidate["segment_length_m"] - candidate["offset_m"],
            ),
        ),
    ]

def rail_anchor_connection(start, end):
    # (məsafə, yol növü, başlanğıc rels ucu, son rels ucu)
    best = (np.inf, 2, -1, -1)

    # Eyni kiçik rels hissəsindədirsə, birbaşa məsafə də mümkündür.
    if start["edge_id"] == end["edge_id"]:
        best = (
            abs(start["offset_m"] - end["offset_m"]),
            0, -1, -1,
        )

    for source_node, source_offset in anchor_endpoints(start):
        for target_node, target_offset in anchor_endpoints(end):
            distance = (
                source_offset
                + endpoint_distances_m[
                    endpoint_matrix_row[source_node], target_node
                ]
                + target_offset
            )

            best = min(
                best,
                (float(distance), 1, source_node, target_node),
            )

    return {
        "length_m": best[0],
        "path_kind": (
            "no_path" if not np.isfinite(best[0])
            else "same_segment" if best[1] == 0
            else "graph"
        ),
        "source_endpoint": best[2],
        "target_endpoint": best[3],
    }

candidates_by_station = {
    station_id: group.to_dict("records")
    for station_id, group in metro_anchor_candidates.groupby(
        "station_id", sort=False
    )
}

rail_candidate_connections = {}
comparison_records = []

for link in metro_ride_links.itertuples(index=False):
    options = []

    for start in candidates_by_station[link.from_station_id]:
        for end in candidates_by_station[link.to_station_id]:
            key = (start["candidate_id"], end["candidate_id"])
            connection = rail_anchor_connection(start, end)
            rail_candidate_connections[key] = connection
            options.append(connection["length_m"])

    comparison_records.append({
        "from_name": link.from_name,
        "to_name": link.to_name,
        "previous_distance_m": link.track_length_m,
        "best_candidate_distance_m": min(options),
    })

rail_candidate_comparison = pd.DataFrame(comparison_records)
display(rail_candidate_comparison.round(2))

,from_name,to_name,previous_distance_m,best_candidate_distance_m
0,Dərnəgül,Azadlıq prospekti,1601.38,1601.38
1,Azadlıq prospekti,Nəsimi,1420.82,1395.27
2,Nəsimi,Memar Əcəmi,2025.13,2025.13
3,Memar Əcəmi,20 Yanvar,912.90,912.90
4,20 Yanvar,İnşaatçılar,1963.76,1963.76
5,İnşaatçılar,Elmlər Akademiyası,2096.24,2073.62
6,Elmlər Akademiyası,Nizami,1326.66,1326.66
7,Nizami,28 May,1778.89,1777.83
8,Xocəsən,Avtovağzal,2338.27,2322.23
9,Avtovağzal,Memar Əcəmi 2,6135.01,2053.49


In [56]:
# Hər xətt boyunca uyğun rels nöqtələrini birlikdə seçirik.
# Məqsəd: xəttin ümumi rels məsafəsini minimum etmək.
# Bərabərlikdə stansiya nöqtəsinə daha yaxın namizədlər seçilir.

line_station_sequences = {
    "green": [
        "Dərnəgül", "Azadlıq prospekti", "Nəsimi",
        "Memar Əcəmi", "20 Yanvar", "İnşaatçılar",
        "Elmlər Akademiyası", "Nizami", "28 May",
    ],
    "purple": [
        "Xocəsən", "Avtovağzal", "Memar Əcəmi 2", "8 Noyabr",
    ],
    "red": [
        "Sahil", "28 May", "Gənclik",
    ],
}

anchor_candidates_by_name = {
    name: group.sort_values("candidate_id").to_dict("records")
    for name, group in metro_anchor_candidates.groupby(
        "station_name", sort=False
    )
}

anchor_by_id = {
    row["candidate_id"]: row
    for row in metro_anchor_candidates.to_dict("records")
}


def get_anchor_connection(start, end):
    key = (start["candidate_id"], end["candidate_id"])

    if key not in rail_candidate_connections:
        rail_candidate_connections[key] = rail_anchor_connection(
            start, end
        )

    return rail_candidate_connections[key]


chosen_anchor_records = []
chosen_link_records = []

for line_id, station_names in line_station_sequences.items():
    # Vəziyyət:
    # (ümumi rels məsafəsi, ümumi yerləşdirmə məsafəsi, namizəd ID-ləri)
    states = {
        candidate["candidate_id"]: (
            0.0,
            float(candidate["point_distance_m"]),
            (candidate["candidate_id"],),
        )
        for candidate in anchor_candidates_by_name[station_names[0]]
    }

    for station_name in station_names[1:]:
        next_states = {}

        for end in anchor_candidates_by_name[station_name]:
            options = []

            for previous_id, score in states.items():
                start = anchor_by_id[previous_id]
                connection = get_anchor_connection(start, end)
                length_m = float(connection["length_m"])

                if not np.isfinite(length_m):
                    continue

                options.append((
                    score[0] + length_m,
                    score[1] + float(end["point_distance_m"]),
                    score[2] + (end["candidate_id"],),
                ))

            if options:
                next_states[end["candidate_id"]] = min(options)

        if not next_states:
            raise ValueError(
                f"{line_id}: {station_name} üçün ardıcıl rels yolu yoxdur."
            )

        states = next_states

    best_score = min(states.values())
    selected_ids = best_score[2]

    for station_name, candidate_id in zip(
        station_names, selected_ids
    ):
        chosen_anchor_records.append({
            **anchor_by_id[candidate_id],
            "line_id": line_id,
            "review_status": "pending_geometry_review",
        })

    for start_id, end_id in zip(
        selected_ids[:-1], selected_ids[1:]
    ):
        start = anchor_by_id[start_id]
        end = anchor_by_id[end_id]
        connection = get_anchor_connection(start, end)

        original = metro_ride_links.loc[
            metro_ride_links["line_id"].eq(line_id)
            & metro_ride_links["from_name"].eq(start["station_name"])
            & metro_ride_links["to_name"].eq(end["station_name"])
        ]

        if len(original) != 1:
            raise ValueError("Metro bağlantısı tək müəyyən edilmədi.")

        chosen_link_records.append({
            "link_id": original.iloc[0]["link_id"],
            "line_id": line_id,
            "from_name": start["station_name"],
            "to_name": end["station_name"],
            "from_candidate_id": start_id,
            "to_candidate_id": end_id,
            "previous_distance_m": float(
                original.iloc[0]["track_length_m"]
            ),
            "consistent_distance_m": float(connection["length_m"]),
            "path_kind": connection["path_kind"],
            "source_endpoint": connection["source_endpoint"],
            "target_endpoint": connection["target_endpoint"],
            "review_status": "pending_geometry_review",
        })

selected_line_anchors = pd.DataFrame(chosen_anchor_records)
metro_line_geometry_plan = pd.DataFrame(chosen_link_records)

if selected_line_anchors.duplicated(
    ["line_id", "station_id"]
).any():
    raise ValueError("Bir xəttdə stansiyaya iki rels nöqtəsi seçilib.")

print("Ardıcıl seçilmiş metro bağlantıları:",
      len(metro_line_geometry_plan))

display(
    metro_line_geometry_plan[[
        "line_id", "from_name", "to_name",
        "previous_distance_m", "consistent_distance_m",
    ]].round(2)
)

Ardıcıl seçilmiş metro bağlantıları: 13


,line_id,from_name,to_name,previous_distance_m,consistent_distance_m
0,green,Dərnəgül,Azadlıq prospekti,1601.38,1601.38
1,green,Azadlıq prospekti,Nəsimi,1420.82,1420.82
2,green,Nəsimi,Memar Əcəmi,2025.13,2025.13
3,green,Memar Əcəmi,20 Yanvar,912.90,912.90
4,green,20 Yanvar,İnşaatçılar,1963.76,1963.76
5,green,İnşaatçılar,Elmlər Akademiyası,2096.24,2096.24
6,green,Elmlər Akademiyası,Nizami,1326.66,1326.66
7,green,Nizami,28 May,1778.89,1777.83
8,purple,Xocəsən,Avtovağzal,2338.27,2322.23
9,purple,Avtovağzal,Memar Əcəmi 2,6135.01,2081.79


In [57]:
import folium
import threading

from functools import partial
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from IPython.display import IFrame, display
from branca.element import Element
from html import escape
from time import time_ns

metro_node_lookup = metro_nodes.set_index("node_id")


def reconstruct_rail_geometry(plan_row):
    start = anchor_by_id[plan_row["from_candidate_id"]]
    end = anchor_by_id[plan_row["to_candidate_id"]]

    coordinates = [
        (float(start["snap_x"]), float(start["snap_y"]))
    ]
    traversed_nodes = []

    if plan_row["path_kind"] == "graph":
        source = int(plan_row["source_endpoint"])
        target = int(plan_row["target_endpoint"])

        predecessor_row = endpoint_predecessors[
            endpoint_matrix_row[source]
        ]

        traversed_nodes = [target]

        while traversed_nodes[-1] != source:
            previous = int(predecessor_row[traversed_nodes[-1]])

            if previous < 0:
                raise ValueError("Rels yolunu geri qurmaq mümkün olmadı.")

            traversed_nodes.append(previous)

            if len(traversed_nodes) > len(metro_nodes):
                raise ValueError("Rels yolunda gözlənilməz dövr yarandı.")

        traversed_nodes.reverse()

        coordinates.extend(
            metro_node_lookup.loc[
                traversed_nodes, ["x", "y"]
            ].itertuples(index=False, name=None)
        )

    elif plan_row["path_kind"] != "same_segment":
        raise ValueError("Bu bağlantı üçün rels yolu yoxdur.")

    coordinates.append(
        (float(end["snap_x"]), float(end["snap_y"]))
    )

    # Ardıcıl təkrarlanan koordinatları çıxarırıq.
    cleaned = [coordinates[0]]

    for coordinate in coordinates[1:]:
        if coordinate != cleaned[-1]:
            cleaned.append(coordinate)

    if len(cleaned) < 2:
        raise ValueError("Metro bağlantısının həndəsəsi sıfır uzunluqludur.")

    geometry = LineString(cleaned)
    expected_length = float(plan_row["consistent_distance_m"])

    if not np.isclose(
        geometry.length, expected_length, rtol=1e-9, atol=1e-6
    ):
        raise ValueError(
            f"{plan_row['from_name']} → {plan_row['to_name']}: "
            "həndəsə ilə hesablanan məsafə uyğun gəlmir."
        )

    assumed_count = (
        int(
            metro_node_lookup.loc[
                traversed_nodes, "endpoint_transition_assumed"
            ].sum()
        )
        if traversed_nodes else 0
    )

    return geometry, assumed_count


geometry_records = []

for record in metro_line_geometry_plan.to_dict("records"):
    geometry, assumed_count = reconstruct_rail_geometry(record)

    geometry_records.append({
        **record,
        "geometry": geometry,
        "assumed_transition_nodes": assumed_count,
    })

metro_review_routes_m = gpd.GeoDataFrame(
    geometry_records,
    geometry="geometry",
    crs=METRIC_CRS,
)

print("Həndəsəsi qurulmuş bağlantı:", len(metro_review_routes_m))
print("Bütün həndəsələrin uzunluğu hesablanan məsafə ilə uyğun gəlir.")

# Xəritə üçün koordinatları enlik/uzunluğa çeviririk.
routes_wgs = metro_review_routes_m.to_crs(4326)

anchors_wgs = gpd.GeoDataFrame(
    selected_line_anchors.copy(),
    geometry=gpd.points_from_xy(
        selected_line_anchors["snap_x"],
        selected_line_anchors["snap_y"],
    ),
    crs=METRIC_CRS,
).to_crs(4326)

line_colors = {
    "green": "#16803c",
    "purple": "#9333ea",
    "red": "#dc2626",
}

metro_review_map = folium.Map(
    location=[40.405, 49.83],
    zoom_start=12,
    tiles="OpenStreetMap",
)

for line_id, color in line_colors.items():
    layer = folium.FeatureGroup(name=f"Metro: {line_id}")

    for row in routes_wgs.loc[
        routes_wgs["line_id"].eq(line_id)
    ].itertuples(index=False):
        is_closure = (
            row.from_name == "Nizami"
            and row.to_name == "28 May"
        )

        tooltip = (
            f"{escape(row.from_name)} → {escape(row.to_name)}"
            f" | {row.consistent_distance_m:.0f} m"
            f" | fərziyyəli keçid düyünü: "
            f"{row.assumed_transition_nodes}"
        )

        folium.PolyLine(
            locations=[
                (y, x) for x, y in row.geometry.coords
            ],
            color=color,
            weight=5,
            opacity=0.85,
            dash_array="8, 8" if is_closure else None,
            tooltip=tooltip,
        ).add_to(layer)

    for row in anchors_wgs.loc[
        anchors_wgs["line_id"].eq(line_id)
    ].itertuples(index=False):
        folium.CircleMarker(
            location=[row.geometry.y, row.geometry.x],
            radius=5,
            color=color,
            fill=True,
            fill_color="white",
            fill_opacity=1,
            tooltip=(
                f"{escape(row.station_name)} ({line_id})"
                f" | stansiya nöqtəsindən "
                f"{row.point_distance_m:.1f} m"
            ),
        ).add_to(layer)

    layer.add_to(metro_review_map)

folium.LayerControl(collapsed=False).add_to(metro_review_map)
metro_review_map.fit_bounds(metro_review_map.get_bounds())

# Notebook daxilində işləyən əvvəlki HTTP üsulu.
metro_review_map.get_root().header.add_child(
    Element(
        '<meta name="referrer" '
        'content="strict-origin-when-cross-origin">'
    ),
    name="citypilot_referrer",
)

map_dir = repo / "outputs"
map_dir.mkdir(parents=True, exist_ok=True)

metro_review_map.save(str(map_dir / "metro_geometry_review.html"))

if "citypilot_map_server" not in globals():
    handler = partial(
        SimpleHTTPRequestHandler,
        directory=str(map_dir.resolve()),
    )
    citypilot_map_server = ThreadingHTTPServer(
        ("127.0.0.1", 0), handler
    )
    threading.Thread(
        target=citypilot_map_server.serve_forever,
        daemon=True,
    ).start()

port = citypilot_map_server.server_address[1]

display(IFrame(
    src=(
        f"http://127.0.0.1:{port}/metro_geometry_review.html"
        f"?v={time_ns()}"
    ),
    width="100%",
    height=650,
))

Həndəsəsi qurulmuş bağlantı: 13
Bütün həndəsələrin uzunluğu hesablanan məsafə ilə uyğun gəlir.


127.0.0.1 - - [17/Oct/2026 16:24:33] "GET /metro_geometry_review.html?v=1792239873424533300 HTTP/1.1" 200 -


In [58]:
from shapely.geometry import Point

review_records = []

for row in metro_review_routes_m.itertuples(index=False):
    geometry = row.geometry

    start = Point(geometry.coords[0])
    end = Point(geometry.coords[-1])
    direct_distance_m = start.distance(end)

    other_stations = selected_line_anchors.loc[
        selected_line_anchors["line_id"].eq(row.line_id)
        & ~selected_line_anchors["station_name"].isin([
            row.from_name, row.to_name
        ])
    ]

    passed_stations = []

    for station in other_stations.itertuples(index=False):
        anchor = Point(station.snap_x, station.snap_y)

        # Yalnız seçilmiş rels nöqtəsinin yolun üzərində olmasını yoxlayır.
        if geometry.distance(anchor) <= 1e-6:
            passed_stations.append(station.station_name)

    review_records.append({
        "from_name": row.from_name,
        "to_name": row.to_name,
        "length_m": geometry.length,
        "detour_ratio": (
            geometry.length / direct_distance_m
            if direct_distance_m > 0 else np.nan
        ),
        "other_stations_on_path": ", ".join(passed_stations),
        "geometry_is_simple": geometry.is_simple,
        "assumed_transition_nodes": row.assumed_transition_nodes,
    })

metro_geometry_audit = pd.DataFrame(review_records)

display(metro_geometry_audit.round(2))

,from_name,to_name,length_m,detour_ratio,other_stations_on_path,geometry_is_simple,assumed_transition_nodes
0,Dərnəgül,Azadlıq prospekti,1601.38,1.00,,True,1
1,Azadlıq prospekti,Nəsimi,1420.82,1.00,,True,1
2,Nəsimi,Memar Əcəmi,2025.13,1.08,,True,0
3,Memar Əcəmi,20 Yanvar,912.90,1.02,,True,0
4,20 Yanvar,İnşaatçılar,1963.76,1.13,,True,0
5,İnşaatçılar,Elmlər Akademiyası,2096.24,1.10,,True,0
6,Elmlər Akademiyası,Nizami,1326.66,1.01,,True,0
7,Nizami,28 May,1777.83,1.13,,True,0
8,Xocəsən,Avtovağzal,2322.23,1.72,,True,1
9,Avtovağzal,Memar Əcəmi 2,2081.79,1.09,,True,0


In [59]:
# AYNA mənbəsindəki çıxış → stansiya uyğunluğu.
pilot_exits = metro_exits_m.loc[
    metro_exits_m["station_name"].isin(pilot_station_names),
    ["station_name", "exit_name"],
].copy()

# Əvvəl yaratdığımız giriş ID-ləri ilə eyni format.
pilot_exits["access_id"] = [
    f"metro_exit:{index}"
    for index in pilot_exits.index
]

# Hər çıxışın piyada qrafındakı düyününü əlavə edirik.
metro_exit_access = pilot_exits.merge(
    access_table[[
        "access_id",
        "node_id",
        "snap_distance_m",
        "on_main_component",
    ]],
    on="access_id",
    how="left",
    validate="one_to_one",
)

missing_access = metro_exit_access.loc[
    metro_exit_access["node_id"].isna(),
    ["station_name", "exit_name", "access_id"],
]

if not missing_access.empty:
    display(missing_access)
    raise ValueError(
        "Bəzi pilot metro çıxışları piyada qrafına qoşulmayıb."
    )

metro_exit_access = metro_exit_access.rename(
    columns={"node_id": "walk_node_id"}
)
metro_exit_access["walk_node_id"] = (
    metro_exit_access["walk_node_id"].astype("int64")
)

# OSM stansiya ID-sini əlavə edirik.
metro_exit_access = metro_exit_access.merge(
    stations_m[["station_name", "station_id"]],
    on="station_name",
    how="left",
    validate="many_to_one",
)

station_access_summary = (
    metro_exit_access.groupby("station_name")
    .agg(
        exit_count=("access_id", "nunique"),
        min_snap_m=("snap_distance_m", "min"),
        max_snap_m=("snap_distance_m", "max"),
        all_on_main_component=("on_main_component", "all"),
    )
    .reindex(pilot_station_names)
)

if station_access_summary["exit_count"].isna().any():
    raise ValueError("Pilot stansiyalardan birinin giriş uyğunluğu yoxdur.")

print("Girişi müəyyən edilmiş stansiya:",
      metro_exit_access["station_id"].nunique())
print("Piyada qrafına bağlı metro çıxışı:", len(metro_exit_access))

display(station_access_summary.round(2))

Girişi müəyyən edilmiş stansiya: 15
Piyada qrafına bağlı metro çıxışı: 53


,exit_count,min_snap_m,max_snap_m,all_on_main_component
station_name,,,,
Dərnəgül,4,4.52,17.03,True
Azadlıq prospekti,4,1.55,8.64,True
Nəsimi,4,1.21,4.30,True
Memar Əcəmi,4,0.14,2.01,True
20 Yanvar,7,0.27,7.06,True
İnşaatçılar,4,6.53,17.45,True
Elmlər Akademiyası,5,0.20,7.86,True
Nizami,1,0.59,0.59,True
Xocəsən,1,7.15,7.15,True


In [60]:
from io import BytesIO
import xml.etree.ElementTree as ET

parking_source_path = (
    "AYNA_Datasets/03_bicycle_parking/mpv-v-mps.-son.kmz"
)

with ZipFile(ayna_zip) as archive:
    parking_kmz_bytes = archive.read(parking_source_path)

with ZipFile(BytesIO(parking_kmz_bytes)) as archive:
    parking_kml = archive.read("doc.kml")

parking_root = ET.fromstring(parking_kml)
kml_ns = {"k": "http://www.opengis.net/kml/2.2"}

# Hər elementin hansı qovluqda olduğunu tapmaq üçün.
parent_map = {
    child: parent
    for parent in parking_root.iter()
    for child in parent
}

parking_records = []

for source_row, placemark in enumerate(
    parking_root.findall(".//k:Placemark", kml_ns)
):
    # LookAt kamera nöqtəsidir; yalnız Point həndəsəsini oxuyuruq.
    point = placemark.find("k:Point", kml_ns)

    if point is None:
        continue

    coordinate_text = point.findtext(
        "k:coordinates", namespaces=kml_ns
    )
    coordinates = coordinate_text.strip().split()

    if len(coordinates) != 1:
        raise ValueError("Point üçün tək koordinat gözlənilirdi.")

    values = coordinates[0].split(",")
    lon, lat = float(values[0]), float(values[1])

    folder_names = []
    parent = parent_map.get(placemark)

    while parent is not None:
        if parent.tag.rsplit("}", 1)[-1] in {"Folder", "Document"}:
            folder_name = parent.findtext(
                "k:name", default="", namespaces=kml_ns
            ).strip()

            if folder_name:
                folder_names.append(folder_name)

        parent = parent_map.get(parent)

    folder_names.reverse()

    parking_records.append({
        "parking_id": f"ayna_parking:{source_row}",
        "source_placemark": source_row,
        "name": placemark.findtext(
            "k:name", default="", namespaces=kml_ns
        ).strip(),
        "folder_path": " / ".join(folder_names),
        "is_demo": "Sightseeing Tour" in folder_names,
        "lon": lon,
        "lat": lat,
        "source_status": "published",
        "operational_status": "unknown",
    })

parking_points_raw = gpd.GeoDataFrame(
    parking_records,
    geometry=gpd.points_from_xy(
        [record["lon"] for record in parking_records],
        [record["lat"] for record in parking_records],
    ),
    crs=4326,
)

parking_points_m = parking_points_raw.to_crs(METRIC_CRS)

parking_points_m["in_work_area"] = (
    parking_points_m.geometry.intersects(routing_area_m.iloc[0])
)

parking_in_area_m = parking_points_m.loc[
    ~parking_points_m["is_demo"]
    & parking_points_m["in_work_area"]
].copy()

print("Mənbədəki Point sayı:", len(parking_points_raw))
print("Turistik nümunə nöqtələri:",
      int(parking_points_raw["is_demo"].sum()))
print("İş sahəsində qalan nöqtə:", len(parking_in_area_m))

display(
    parking_points_m.groupby(
        ["folder_path", "is_demo"], dropna=False
    ).size().rename("point_count").to_frame()
)

Mənbədəki Point sayı: 844
Turistik nümunə nöqtələri: 10
İş sahəsində qalan nöqtə: 639


,,point_count
folder_path,is_demo,
Temporary Places / Temporary Places / velosiped məntəqələri.11 sentyabr.kmz / velosiped məntəqələri yeni,False,185
Temporary Places / Temporary Places / velosiped məntəqələri.11 sentyabr.kmz / Ümumi 12 iyul. SON olan.kmz / My Places,False,649
Temporary Places / Temporary Places / velosiped məntəqələri.11 sentyabr.kmz / Ümumi 12 iyul. SON olan.kmz / My Places / Sightseeing Tour,True,10


In [61]:
from scipy.spatial import cKDTree

# Tam eyni koordinatdakı mənbə qeydləri.
parking_duplicate_mask = parking_in_area_m.duplicated(
    ["lon", "lat"], keep=False
)

parking_exact_duplicates = parking_in_area_m.loc[
    parking_duplicate_mask,
    ["parking_id", "name", "lon", "lat", "folder_path"],
].sort_values(["lon", "lat", "parking_id"])

print("İş sahəsində mənbə qeydi:", len(parking_in_area_m))
print(
    "Fərqli koordinat sayı:",
    len(parking_in_area_m.drop_duplicates(["lon", "lat"])),
)
print("Eyni koordinat qrupuna düşən qeyd:",
      int(parking_duplicate_mask.sum()))

display(parking_exact_duplicates.head(20))

# 5 metr yalnız baxış məsafəsidir; heç nəyi birləşdirmirik.
PARKING_REVIEW_DISTANCE_M = 5.0

parking_review_points = parking_in_area_m.reset_index(drop=True)

parking_xy = np.column_stack([
    parking_review_points.geometry.x,
    parking_review_points.geometry.y,
])

parking_tree = cKDTree(parking_xy)
nearby_pairs = sorted(
    parking_tree.query_pairs(r=PARKING_REVIEW_DISTANCE_M)
)

nearby_records = []

for left, right in nearby_pairs:
    first = parking_review_points.iloc[left]
    second = parking_review_points.iloc[right]

    nearby_records.append({
        "first_id": first["parking_id"],
        "first_name": first["name"],
        "second_id": second["parking_id"],
        "second_name": second["name"],
        "distance_m": float(
            np.linalg.norm(parking_xy[left] - parking_xy[right])
        ),
    })

parking_nearby_review = pd.DataFrame(
    nearby_records,
    columns=[
        "first_id", "first_name",
        "second_id", "second_name", "distance_m",
    ],
).sort_values(["distance_m", "first_id", "second_id"])

print("5 metr daxilində qeyd cütləri:", len(parking_nearby_review))
display(parking_nearby_review.head(20).round(3))

İş sahəsində mənbə qeydi: 639
Fərqli koordinat sayı: 639
Eyni koordinat qrupuna düşən qeyd: 0


,parking_id,name,lon,lat,folder_path


5 metr daxilində qeyd cütləri: 53


,first_id,first_name,second_id,second_name,distance_m
2,ayna_parking:71,MPS-17,ayna_parking:784,MPV-81-2,1.070
41,ayna_parking:279,MPS-226,ayna_parking:879,MPV-176-2,1.107
18,ayna_parking:140,MPS-86,ayna_parking:717,MPV-14-2,1.507
37,ayna_parking:239,MPS-186,ayna_parking:814,MPV-111-2,1.531
36,ayna_parking:223,MPS-170,ayna_parking:772,MPV-69-2,1.663
48,ayna_parking:471,MPS-418,ayna_parking:781,MPV-78-2,1.843
22,ayna_parking:144,MPS-90,ayna_parking:735,MPV-32-2,1.865
7,ayna_parking:117,MPS-63,ayna_parking:786,MPV-83-2,2.022
29,ayna_parking:171,MPS-117,ayna_parking:704,MPV-1-2,2.124
42,ayna_parking:282,MPS-229,ayna_parking:872,MPV-169-1,2.134


In [62]:
# Girişlər əlavə edildikdən sonrakı piyada komponentləri.
_, parking_walk_components = connected_components(
    walk_graph_access_m,
    directed=False,
    return_labels=True,
)

active_walk_nodes = np.unique(
    walk_arcs_access[["u", "v"]].to_numpy().ravel()
)

parking_main_component = int(
    pd.Series(parking_walk_components[active_walk_nodes])
    .value_counts()
    .idxmax()
)

# Yalnız yol hissələri; metro/terminal giriş bağlantıları daxil deyil.
walk_edge_ids = set(walk_arcs_access["edge_id"])
bike_edge_ids = set(bike_arcs_access["edge_id"])

parking_road_edges = edges_access.loc[
    edges_access["edge_type"].eq("road")
    & edges_access["edge_id"].isin(walk_edge_ids)
].copy()

parking_road_segments_m = gpd.GeoDataFrame(
    parking_road_edges[["edge_id", "u", "v"]].copy(),
    geometry=[
        LineString([(row.x0, row.y0), (row.x1, row.y1)])
        for row in parking_road_edges.itertuples(index=False)
    ],
    crs=METRIC_CRS,
).reset_index(drop=True)

parking_road_segments_m["walk_component"] = (
    parking_walk_components[
        parking_road_segments_m["u"].to_numpy(dtype="int64")
    ]
)


def review_parking_snap(segments, mode):
    matches = gpd.sjoin_nearest(
        parking_in_area_m[["parking_id", "name", "geometry"]],
        segments[["edge_id", "walk_component", "geometry"]],
        how="left",
        distance_col=f"{mode}_snap_m",
    )

    # Eyni məsafəli namizədlərdə sabit seçim.
    matches = (
        matches.sort_values([
            "parking_id", f"{mode}_snap_m", "edge_id"
        ])
        .drop_duplicates("parking_id")
    )

    matches[f"{mode}_on_main"] = (
        matches["walk_component"].eq(parking_main_component)
    )

    return matches[[
        "parking_id", "edge_id",
        f"{mode}_snap_m", f"{mode}_on_main",
    ]].rename(columns={"edge_id": f"{mode}_edge_id"})


parking_walk_snap = review_parking_snap(
    parking_road_segments_m, "walk"
)

parking_bike_snap = review_parking_snap(
    parking_road_segments_m.loc[
        parking_road_segments_m["edge_id"].isin(bike_edge_ids)
    ],
    "bike",
)

parking_snap_review = (
    parking_in_area_m[["parking_id", "name"]]
    .merge(parking_walk_snap, on="parking_id", validate="one_to_one")
    .merge(parking_bike_snap, on="parking_id", validate="one_to_one")
)

print("Yoxlanan parkinq qeydi:", len(parking_snap_review))

display(
    parking_snap_review[["walk_snap_m", "bike_snap_m"]]
    .describe(percentiles=[0.5, 0.9, 0.95, 0.99])
    .round(2)
)

print("Ən uzaq velo yolu olan 10 nöqtə:")

display(
    parking_snap_review.sort_values(
        ["bike_snap_m", "parking_id"],
        ascending=[False, True],
    ).head(10).round(2)
)

Yoxlanan parkinq qeydi: 639


,walk_snap_m,bike_snap_m
count,639.00,639.00
mean,5.38,10.68
std,4.26,8.92
min,0.01,0.01
50%,4.45,8.99
90%,11.19,19.31
95%,12.72,24.83
99%,18.23,41.91
max,35.07,96.06


Ən uzaq velo yolu olan 10 nöqtə:


,parking_id,name,walk_edge_id,walk_snap_m,walk_on_main,bike_edge_id,bike_snap_m,bike_on_main
340,ayna_parking:516,MPS-463,roadrow:93428:segment:13,3.34,True,roadrow:184875:segment:3,96.06,True
203,ayna_parking:258,MPS-205,roadrow:101656:segment:0,3.07,True,roadrow:3702:segment:0,66.97,True
337,ayna_parking:512,MPS-459,roadrow:93405:segment:0,1.60,True,roadrow:75345:segment:0,63.61,True
338,ayna_parking:513,MPS-460,roadrow:93403:segment:1,6.54,True,roadrow:64944:segment:22,61.28,True
178,ayna_parking:233,MPS-180,roadrow:93361:segment:7,4.10,True,roadrow:79739:segment:1,60.19,True
627,ayna_parking:869,MPV-166-2,roadrow:234101:segment:7,4.82,True,roadrow:336680:segment:1,59.38,True
316,ayna_parking:390,MPS-337,roadrow:93240:segment:0,6.91,True,roadrow:93222:segment:3,42.11,True
326,ayna_parking:472,MPS-419,roadrow:93384:segment:22,3.63,True,roadrow:64944:segment:27,41.57,True
464,ayna_parking:698,MPS-645,roadrow:93448:segment:10,2.94,True,roadrow:312753:segment:3,40.49,True
624,ayna_parking:866,MPV-163-2,roadrow:312483:segment:2,2.42,True,roadrow:72611:segment:4,39.85,True


In [63]:
parking_source_lookup = parking_in_area_m.set_index("parking_id")
parking_segment_lookup = parking_road_segments_m.set_index("edge_id")

nearby_review_ids = (
    set(parking_nearby_review["first_id"])
    | set(parking_nearby_review["second_id"])
)

parking_plan_records = []

for match in parking_walk_snap.itertuples(index=False):
    source = parking_source_lookup.loc[match.parking_id]
    segment = parking_segment_lookup.loc[match.walk_edge_id]
    line = segment.geometry

    # Nöqtənin yol hissəsi üzərindəki ən yaxın yeri.
    offset_m = float(line.project(source.geometry))
    snapped = line.interpolate(offset_m)

    parking_plan_records.append({
        "parking_id": match.parking_id,
        "access_id": f"parking_access:{match.parking_id}",
        "label": f"Parkinq — {source['name']}",
        "edge_id": match.walk_edge_id,
        "offset_m": offset_m,
        "fraction": offset_m / line.length,
        "snap_distance_m": source.geometry.distance(snapped),
        "source_x": source.geometry.x,
        "source_y": source.geometry.y,
        "snap_x": snapped.x,
        "snap_y": snapped.y,
        "on_main_component": bool(match.walk_on_main),
        "nearby_record_review": match.parking_id in nearby_review_ids,
        "operational_status": "unknown",
        "connector_status": "assumption",
        "parking_transition_enabled": False,
    })

parking_snap_plan = pd.DataFrame(parking_plan_records)

print("Qoşulması planlaşdırılan parkinq nöqtəsi:",
      len(parking_snap_plan))
print("Toxunulan fərqli yol hissəsi:",
      parking_snap_plan["edge_id"].nunique())
print("Əsas piyada komponentindən kənarda:",
      int((~parking_snap_plan["on_main_component"]).sum()))
print("Ən uzun giriş bağlantısı, metr:",
      round(parking_snap_plan["snap_distance_m"].max(), 2))

display(
    parking_snap_plan.sort_values(
        ["snap_distance_m", "parking_id"],
        ascending=[False, True],
    )[[
        "parking_id", "label", "snap_distance_m",
        "on_main_component", "nearby_record_review",
        "parking_transition_enabled",
    ]].head(10).round(2)
)


Qoşulması planlaşdırılan parkinq nöqtəsi: 639
Toxunulan fərqli yol hissəsi: 569
Əsas piyada komponentindən kənarda: 5
Ən uzun giriş bağlantısı, metr: 35.07


,parking_id,label,snap_distance_m,on_main_component,nearby_record_review,parking_transition_enabled
216,ayna_parking:331,Parkinq — MPS-278,35.07,True,False,False
115,ayna_parking:215,Parkinq — MPS-162,23.15,True,True,False
481,ayna_parking:741,Parkinq — MPV-38-2,21.05,True,False,False
514,ayna_parking:771,Parkinq — MPV-68-2,20.91,True,True,False
298,ayna_parking:526,Parkinq — MPS-473,20.08,True,False,False
317,ayna_parking:546,Parkinq — MPS-493,19.30,True,False,False
7,ayna_parking:107,Parkinq — MPS-53,18.55,True,False,False
506,ayna_parking:764,Parkinq — MPV-61-2,17.71,True,False,False
226,ayna_parking:341,Parkinq — MPS-288,17.61,True,False,False
477,ayna_parking:738,Parkinq — MPV-35-2,17.03,True,False,False


In [64]:
# Bu rəqəm yalnız hesablama dəqiqliyi üçündür.
PARKING_SPLIT_EPS_M = 1e-8

if not np.array_equal(
    np.sort(nodes_access["node_id"].to_numpy()),
    np.arange(len(nodes_access)),
):
    raise ValueError("Düyün ID-ləri ardıcıl deyil.")

edge_lookup = edges_access.set_index("edge_id")

new_parking_road_nodes = []
parking_child_edges = []
parking_anchor_map = {}
replaced_parking_edges = set()
parking_length_errors = []

next_node_id = len(nodes_access)

for edge_id, group in parking_snap_plan.groupby(
    "edge_id", sort=True
):
    original = edge_lookup.loc[edge_id].to_dict()
    original["edge_id"] = edge_id

    line = parking_segment_lookup.loc[edge_id].geometry
    length_m = float(line.length)

    # Kəsimlər: yolun başlanğıcı, daxili nöqtələri və sonu.
    cuts = [(0.0, int(original["u"]))]
    previous_internal_offset = None
    previous_internal_node = None

    for point in group.sort_values(
        ["offset_m", "parking_id"]
    ).itertuples(index=False):
        offset = float(point.offset_m)

        if (
            offset < -PARKING_SPLIT_EPS_M
            or offset > length_m + PARKING_SPLIT_EPS_M
        ):
            raise ValueError("Qoşulma nöqtəsi yol hissəsindən kənardadır.")

        offset = min(max(offset, 0.0), length_m)

        if offset <= PARKING_SPLIT_EPS_M:
            anchor_node = int(original["u"])

        elif length_m - offset <= PARKING_SPLIT_EPS_M:
            anchor_node = int(original["v"])

        elif (
            previous_internal_offset is not None
            and offset - previous_internal_offset
            <= PARKING_SPLIT_EPS_M
        ):
            # Eyni qoşulma yerində əlavə düyün yaratmırıq.
            anchor_node = previous_internal_node

        else:
            location = line.interpolate(offset)
            anchor_node = next_node_id
            next_node_id += 1

            new_parking_road_nodes.append({
                "node_id": anchor_node,
                "x": location.x,
                "y": location.y,
                "endpoint_transition_assumed": False,
            })

            cuts.append((offset, anchor_node))
            previous_internal_offset = offset
            previous_internal_node = anchor_node

        parking_anchor_map[point.parking_id] = anchor_node

    cuts.append((length_m, int(original["v"])))

    # Hamısı mövcud uc nöqtələrə düşübsə, yol dəyişmir.
    if len(cuts) == 2:
        continue

    replaced_parking_edges.add(edge_id)
    child_length_sum = 0.0

    for part_number, (left, right) in enumerate(
        zip(cuts[:-1], cuts[1:])
    ):
        start_offset, u = left
        end_offset, v = right

        start = line.interpolate(start_offset)
        end = line.interpolate(end_offset)
        child_length = float(start.distance(end))

        if child_length <= 0:
            raise ValueError("Sıfır uzunluqlu yol hissəsi yarandı.")

        # Bütün əvvəlki icazə və mənbə sahələrini saxlayırıq.
        child = original.copy()
        child.update({
            "edge_id": f"{edge_id}:parking_part:{part_number}",
            "u": u,
            "v": v,
            "x0": start.x,
            "y0": start.y,
            "x1": end.x,
            "y1": end.y,
            "length_m": child_length,
        })

        parking_child_edges.append(child)
        child_length_sum += child_length

    parking_length_errors.append(
        abs(child_length_sum - float(original["length_m"]))
    )

nodes_parking_split = pd.concat(
    [
        nodes_access,
        pd.DataFrame(
            new_parking_road_nodes,
            columns=[
                "node_id", "x", "y",
                "endpoint_transition_assumed",
            ],
        ),
    ],
    ignore_index=True,
)

edges_parking_split = pd.concat(
    [
        edges_access.loc[
            ~edges_access["edge_id"].isin(replaced_parking_edges)
        ],
        pd.DataFrame(parking_child_edges),
    ],
    ignore_index=True,
)

parking_access_anchors = parking_snap_plan.copy()
parking_access_anchors["anchor_node_id"] = (
    parking_access_anchors["parking_id"].map(parking_anchor_map)
)

if parking_access_anchors["anchor_node_id"].isna().any():
    raise ValueError("Parkinq üçün qoşulma düyünü tapılmadı.")

parking_access_anchors["anchor_node_id"] = (
    parking_access_anchors["anchor_node_id"].astype("int64")
)

if edges_parking_split["edge_id"].duplicated().any():
    raise ValueError("Təkrarlanan yol hissəsi ID-si yarandı.")

max_length_error = max(parking_length_errors, default=0.0)

if max_length_error > 1e-6:
    raise ValueError("Bölünmə zamanı yol uzunluğu dəyişdi.")

print("Yeni yol düyünləri:", len(new_parking_road_nodes))
print("Bölünmədən sonra düyün sayı:", len(nodes_parking_split))
print("Bölünmədən sonra yol hissələri:", len(edges_parking_split))
print("Maksimum uzunluq fərqi, metr:", max_length_error)
print("Qoşulma düyünü müəyyən edilmiş parkinq:",
      len(parking_access_anchors))

Yeni yol düyünləri: 610
Bölünmədən sonra düyün sayı: 119071
Bölünmədən sonra yol hissələri: 131704
Maksimum uzunluq fərqi, metr: 5.684341886080802e-14
Qoşulma düyünü müəyyən edilmiş parkinq: 639


In [65]:
# Əvvəl yoxladığımız piyada girişinin icazələrini nümunə götürürük.
access_templates = edges_access.loc[
    edges_access["edge_id"].str.startswith("access_link:")
]

if access_templates.empty:
    raise ValueError("Piyada giriş bağlantısı nümunəsi tapılmadı.")

connector_template = access_templates.iloc[0].to_dict()
parking_node_lookup = nodes_parking_split.set_index("node_id")

parking_new_nodes = []
parking_connector_edges = []
parking_node_map = {}

next_node_id = len(nodes_parking_split)

for point in parking_access_anchors.itertuples(index=False):
    anchor = parking_node_lookup.loc[point.anchor_node_id]

    # Birləşdirəcəyimiz düyünün faktiki koordinatından istifadə edirik.
    distance_m = float(np.hypot(
        point.source_x - anchor["x"],
        point.source_y - anchor["y"],
    ))

    if distance_m <= PARKING_SPLIT_EPS_M:
        parking_node_map[point.parking_id] = point.anchor_node_id
        continue

    parking_node_id = next_node_id
    next_node_id += 1

    parking_new_nodes.append({
        "node_id": parking_node_id,
        "x": point.source_x,
        "y": point.source_y,
        "endpoint_transition_assumed": False,
    })

    connector = connector_template.copy()
    connector.update({
        "edge_id": f"parking_access_link:{point.parking_id}",
        "parent_edge_id": None,
        "access_id": point.access_id,
        "parking_id": point.parking_id,
        "u": parking_node_id,
        "v": int(point.anchor_node_id),
        "x0": point.source_x,
        "y0": point.source_y,
        "x1": float(anchor["x"]),
        "y1": float(anchor["y"]),
        "length_m": distance_m,
        "edge_type": "access",
        "fclass": "access_connector",
        "source_status": "assumption",
        "snapped_edge_id": point.edge_id,
    })

    parking_connector_edges.append(connector)
    parking_node_map[point.parking_id] = parking_node_id

nodes_parking = pd.concat(
    [
        nodes_parking_split,
        pd.DataFrame(
            parking_new_nodes,
            columns=[
                "node_id", "x", "y",
                "endpoint_transition_assumed",
            ],
        ),
    ],
    ignore_index=True,
)

edges_parking = pd.concat(
    [
        edges_parking_split,
        pd.DataFrame(parking_connector_edges),
    ],
    ignore_index=True,
)

parking_table = parking_access_anchors.copy()
parking_table["node_id"] = (
    parking_table["parking_id"].map(parking_node_map)
)

if parking_table["node_id"].isna().any():
    raise ValueError("Parkinqın giriş düyünü müəyyən edilmədi.")

parking_table["node_id"] = parking_table["node_id"].astype("int64")

# Üç rejimin qrafını yeniləyirik.
walk_graph_parking_m, walk_arcs_parking = build_distance_graph(
    edges_parking, len(nodes_parking), "walk"
)

bike_graph_parking_m, bike_arcs_parking = build_distance_graph(
    edges_parking, len(nodes_parking), "bike"
)

drive_graph_parking_m, drive_arcs_parking = build_distance_graph(
    edges_parking, len(nodes_parking), "drive"
)

# Əvvəlki girişlərə olan məsafələr dəyişməməlidir.
parking_terminal_distances_m = dijkstra(
    walk_graph_parking_m,
    directed=True,
    indices=terminal_node_id,
)

previous_access_nodes = access_table["node_id"].to_numpy(
    dtype="int64"
)

before = terminal_walk_distances_m[previous_access_nodes]
after = parking_terminal_distances_m[previous_access_nodes]

if not np.array_equal(np.isfinite(before), np.isfinite(after)):
    raise ValueError("Əvvəlki girişlərin əlçatanlığı dəyişdi.")

finite = np.isfinite(before)

if not np.allclose(
    before[finite], after[finite], rtol=1e-9, atol=1e-6
):
    raise ValueError("Əvvəlki girişlərə olan məsafələr dəyişdi.")

print("Yeni parkinq giriş düyünləri:", len(parking_new_nodes))
print("Əlavə edilmiş piyada giriş bağlantıları:",
      len(parking_connector_edges))
print("Ümumi düyün sayı:", len(nodes_parking))
print("Piyada qrafında istiqamətli bağlantı:",
      walk_graph_parking_m.nnz)
print("Əvvəlki 70 girişin əlçatanlığı və məsafələri qorunub.")
print("Aktiv parkinq keçidi:",
      int(parking_table["parking_transition_enabled"].sum()))

Yeni parkinq giriş düyünləri: 639
Əlavə edilmiş piyada giriş bağlantıları: 639
Ümumi düyün sayı: 119710
Piyada qrafında istiqamətli bağlantı: 258376
Əvvəlki 70 girişin əlçatanlığı və məsafələri qorunub.
Aktiv parkinq keçidi: 0


In [66]:
metro_station_ids = stations_m["station_id"].tolist()

metro_station_index = {
    station_id: index
    for index, station_id in enumerate(metro_station_ids)
}

# Mövcud bağlantılarımız hər iki istiqamətdə işləyir.
if not metro_links["bidirectional"].all():
    raise ValueError("İstiqamət qaydası ayrıca işlənməlidir.")


def build_metro_topology(enabled_column):
    enabled = metro_links.loc[
        metro_links[enabled_column]
    ]

    connections = []

    for link in enabled.itertuples(index=False):
        u = metro_station_index[link.from_station_id]
        v = metro_station_index[link.to_station_id]

        connections.extend([
            {"u": u, "v": v},
            {"u": v, "v": u},
        ])

    arcs = pd.DataFrame(
        connections, columns=["u", "v"]
    ).drop_duplicates(["u", "v"])

    graph = csr_matrix(
        (
            np.ones(len(arcs), dtype="int8"),
            (
                arcs["u"].to_numpy(dtype="int64"),
                arcs["v"].to_numpy(dtype="int64"),
            ),
        ),
        shape=(len(metro_station_ids), len(metro_station_ids)),
    )

    return graph, arcs


metro_normal_topology, metro_normal_topology_arcs = (
    build_metro_topology("normal_enabled")
)

metro_disrupted_topology, metro_disrupted_topology_arcs = (
    build_metro_topology("disrupted_enabled")
)

normal_pairs = set(
    metro_normal_topology_arcs.itertuples(index=False, name=None)
)
disrupted_pairs = set(
    metro_disrupted_topology_arcs.itertuples(index=False, name=None)
)

station_id_by_name = stations_m.set_index(
    "station_name"
)["station_id"].to_dict()

nizami_index = metro_station_index[
    station_id_by_name["Nizami"]
]
may28_index = metro_station_index[
    station_id_by_name["28 May"]
]

expected_removed_pairs = {
    (nizami_index, may28_index),
    (may28_index, nizami_index),
}

if (
    normal_pairs - disrupted_pairs != expected_removed_pairs
    or disrupted_pairs - normal_pairs
):
    raise ValueError("Bağlanma nəzərdə tutulan əlaqədən fərqlidir.")

normal_component_count, normal_labels = connected_components(
    metro_normal_topology,
    directed=False,
)

disrupted_component_count, disrupted_labels = connected_components(
    metro_disrupted_topology,
    directed=False,
)

metro_closure_review = stations_m[[
    "station_id", "station_name"
]].copy()

metro_closure_review["normal_component"] = normal_labels
metro_closure_review["disrupted_component"] = disrupted_labels

print("Hər iki vəziyyətdə stansiya sayı:", len(metro_station_ids))
print("Normal vəziyyətdə metro komponentləri:",
      normal_component_count)
print("Bağlanmada metro komponentləri:",
      disrupted_component_count)
print("Bağlanan istiqamətli əlaqə:", len(expected_removed_pairs))

display(
    metro_closure_review.groupby("disrupted_component")
    .agg(
        station_count=("station_id", "size"),
        stations=("station_name", list),
    )
)

Hər iki vəziyyətdə stansiya sayı: 15
Normal vəziyyətdə metro komponentləri: 1
Bağlanmada metro komponentləri: 2
Bağlanan istiqamətli əlaqə: 2


,station_count,stations
disrupted_component,,
0,12,"[Dərnəgül, Azadlıq prospekti, Nəsimi, Memar Əc..."
1,3,"[28 May, Sahil, Gənclik]"


In [67]:
work_area_geometry = routing_area_m.iloc[0]

# Pilot stansiyaların AYNA çıxışları.
pilot_exit_geometries = metro_exits_m.loc[
    metro_exits_m["station_name"].isin(pilot_station_names)
].copy()

pilot_exit_geometries["inside_work_area"] = (
    pilot_exit_geometries.geometry.apply(
        work_area_geometry.covers
    )
)

# Terminalın faktiki giriş nöqtəsi.
terminal_node = nodes_parking.set_index("node_id").loc[
    terminal_node_id
]
terminal_geometry = Point(
    terminal_node["x"], terminal_node["y"]
)

# Seçilmiş metro yollarının sahədən kənarda qalan uzunluğu.
metro_boundary_review = metro_review_routes_m[[
    "line_id", "from_name", "to_name", "geometry"
]].copy()

metro_boundary_review["outside_length_m"] = (
    metro_boundary_review.geometry.apply(
        lambda geometry: geometry.difference(
            work_area_geometry
        ).length
    )
)

# Bu yolları əvvəl bütöv saxlamışdıq, sərhəddə kəsməmişdik.
boundary_road_count = int(
    roads_m.geometry.intersects(
        work_area_geometry.boundary
    ).sum()
)

outside_exits = pilot_exit_geometries.loc[
    ~pilot_exit_geometries["inside_work_area"],
    ["station_name", "exit_name"],
]

outside_metro_links = metro_boundary_review.loc[
    metro_boundary_review["outside_length_m"] > 1e-6,
    ["line_id", "from_name", "to_name", "outside_length_m"],
]

print("Pilot çıxış sayı:", len(pilot_exit_geometries))
print("Sahədən kənarda qalan pilot çıxış:", len(outside_exits))
print("Terminal sahənin içindədir:",
      work_area_geometry.covers(terminal_geometry))
print("Sahədən kənara çıxan seçilmiş metro bağlantısı:",
      len(outside_metro_links))
print("Sərhədə toxunan və bütöv saxlanmış yol obyekti:",
      boundary_road_count)

if not outside_exits.empty:
    display(outside_exits)

if not outside_metro_links.empty:
    display(outside_metro_links.round(2))

Pilot çıxış sayı: 53
Sahədən kənarda qalan pilot çıxış: 0
Terminal sahənin içindədir: True
Sahədən kənara çıxan seçilmiş metro bağlantısı: 0
Sərhədə toxunan və bütöv saxlanmış yol obyekti: 311


In [68]:
import json
from scipy.sparse import save_npz

checkpoint_dir = repo / "outputs" / "c2_network_checkpoint"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Məsafə qrafları.
save_npz(checkpoint_dir / "walk_graph_m.npz", walk_graph_parking_m)
save_npz(checkpoint_dir / "bike_graph_m.npz", bike_graph_parking_m)
save_npz(checkpoint_dir / "drive_graph_m.npz", drive_graph_parking_m)

# Düyünlər, yol hissələri və giriş uyğunluqları.
checkpoint_tables = {
    "nodes": nodes_parking,
    "edges": edges_parking,
    "parking_access": parking_table,
    "metro_exit_access": metro_exit_access,
    "metro_links": metro_links,
}

for name, table in checkpoint_tables.items():
    table.to_csv(
        checkpoint_dir / f"{name}.csv.gz",
        index=False,
        compression="gzip",
        encoding="utf-8",
    )

# Seçilmiş metro həndəsələri və stansiya nöqtələri.
(checkpoint_dir / "metro_routes.geojson").write_text(
    metro_review_routes_m.to_crs(4326).to_json(
        na="null", drop_id=True
    ),
    encoding="utf-8",
)

(checkpoint_dir / "metro_stations.geojson").write_text(
    stations_m.to_crs(4326).to_json(
        na="null", drop_id=True
    ),
    encoding="utf-8",
)

checkpoint_summary = {
    "status": "notebook_network_checkpoint",
    "metric_crs": METRIC_CRS,
    "nodes": int(len(nodes_parking)),
    "edges": int(len(edges_parking)),
    "directed_connections": {
        "walk": int(walk_graph_parking_m.nnz),
        "bike": int(bike_graph_parking_m.nnz),
        "drive": int(drive_graph_parking_m.nnz),
    },
    "metro_stations": int(len(stations_m)),
    "metro_ride_links": int(len(metro_review_routes_m)),
    "metro_exits": int(len(metro_exit_access)),
    "parking_records": int(len(parking_table)),
    "enabled_parking_transitions": int(
        parking_table["parking_transition_enabled"].sum()
    ),
    "closure": {
        "from": "Nizami",
        "to": "28 May",
        "both_directions": True,
        "stations_remain_open": True,
    },
    "open_items": [
        "Assumed grade transitions need review.",
        "Parking operational status remains unknown.",
        "Full bus geometries need boundary checks after handoff.",
        "Clean-kernel execution and module export remain to be checked.",
    ],
}

(checkpoint_dir / "summary.json").write_text(
    json.dumps(
        checkpoint_summary,
        ensure_ascii=False,
        indent=2,
        allow_nan=False,
    ),
    encoding="utf-8",
)

print("Checkpoint saxlanıldı:", checkpoint_dir)
print("Növbəti mərhələ: C3 — səfər vaxtı və nəqliyyat keçidləri.")

Checkpoint saxlanıldı: c:\Users\celal\OneDrive\Desktop\ayna-hackathon-2026\ayna-hackathon-2026\outputs\c2_network_checkpoint
Növbəti mərhələ: C3 — səfər vaxtı və nəqliyyat keçidləri.


## C3 — Səfər vaxtı və nəqliyyat keçidləri

Məsafələri vaxta çeviririk, metro/avtobus gözləməsini və
nəqliyyat dəyişməsini modelləşdiririk.
Normal və bağlanma vəziyyətlərində səfərləri müqayisə edirik.

In [69]:
import os

# Mühit və böyük nəticə qovluqlarını axtarmırıq.
skip_directories = {
    ".git", ".venv", "venv", "env", "citypilot-env",
    "__pycache__", "node_modules", "outputs", "raw",
}

wanted_files = {
    "assumptions.yaml",
    "limits.yaml",
    "evaluation.yaml",
    "closure.yaml",
    "bus_services.csv",
    "bus_patterns.csv",
    "bus_stops.csv",
}

c3_input_paths = []

for directory, subdirectories, filenames in os.walk(repo):
    subdirectories[:] = [
        name for name in subdirectories
        if name not in skip_directories
    ]

    for filename in filenames:
        if filename in wanted_files:
            c3_input_paths.append(Path(directory) / filename)

c3_input_paths.sort()

print("C3 üçün tapılan fayllar:")

if c3_input_paths:
    for path in c3_input_paths:
        print("-", path.relative_to(repo))
else:
    print("Parametr və avtobus faylları hələ bu checkout-da yoxdur.")

# Əlinin əsas fərziyyələr faylı.
assumptions_path = repo / "config" / "assumptions.yaml"

if assumptions_path.exists():
    print("\n--- config/assumptions.yaml ---")
    print(assumptions_path.read_text(encoding="utf-8-sig"))

# Tək nüsxə varsa avtobus sütunlarını yoxlayırıq.
bus_service_paths = [
    path for path in c3_input_paths
    if path.name == "bus_services.csv"
]

if len(bus_service_paths) == 1:
    bus_services_raw = pd.read_csv(bus_service_paths[0])

    print("\nAvtobus xidmətlərinin sütunları:")
    print(bus_services_raw.columns.tolist())

    display(bus_services_raw.head(10))

elif len(bus_service_paths) > 1:
    print("\nbus_services.csv-nin bir neçə nüsxəsi var.")

C3 üçün tapılan fayllar:
Parametr və avtobus faylları hələ bu checkout-da yoxdur.


In [70]:
for relative_path in [
    "tools/export_notebooks.py",
    "src/citypilot/routing_engine.py",
]:
    path = repo / relative_path
    print(relative_path, "→", "VAR" if path.exists() else "YOXDUR")

tools/export_notebooks.py → VAR
src/citypilot/routing_engine.py → VAR


In [71]:
curated_dir = repo / "data" / "curated"

bus_services = pd.read_csv(
    curated_dir / "bus_services.csv",
    dtype={"service_group_id": "string", "route": "string"},
)

bus_patterns = pd.read_csv(
    curated_dir / "bus_patterns.csv",
    dtype={
        "route": "string",
        "direction": "string",
        "pattern_id": "string",
        "stop_id": "string",
    },
)

bus_stops = pd.read_csv(
    curated_dir / "bus_stops.csv",
    dtype={"stop_id": "string"},
)

# Hesablamada istifadə edəcəyimiz rəqəmlər.
for column in [
    "speed_kmh", "dwell_s", "layover_min",
    "headway_default_min", "cycle_km_geometry",
]:
    bus_services[column] = pd.to_numeric(
        bus_services[column], errors="raise"
    )

    if not np.isfinite(bus_services[column]).all():
        raise ValueError(f"{column}: boş və ya sonsuz dəyər var.")

for column in ["seq", "chainage_m", "segment_m", "snap_m"]:
    bus_patterns[column] = pd.to_numeric(
        bus_patterns[column], errors="raise"
    )

for column in ["lat", "lon"]:
    bus_stops[column] = pd.to_numeric(
        bus_stops[column], errors="raise"
    )

if bus_stops["stop_id"].isna().any() or bus_stops["stop_id"].duplicated().any():
    raise ValueError("Dayanacaq ID-ləri boşdur və ya təkrarlanır.")

if bus_services["service_group_id"].duplicated().any():
    raise ValueError("Servis qrupu ID-si təkrarlanır.")

if bus_patterns[["pattern_id", "seq"]].duplicated().any():
    raise ValueError("Bir istiqamətdə dayanacaq sırası təkrarlanır.")

missing_stop_ids = (
    set(bus_patterns["stop_id"]) - set(bus_stops["stop_id"])
)

if missing_stop_ids:
    raise ValueError(f"Koordinatı olmayan dayanacaqlar: {missing_stop_ids}")

if (
    (bus_services["speed_kmh"] <= 0).any()
    or (bus_services["headway_default_min"] <= 0).any()
    or (bus_services["dwell_s"] < 0).any()
    or (bus_services["layover_min"] < 0).any()
):
    raise ValueError("Sürət, interval və ya dayanma dəyəri uyğun deyil.")

if (
    ~np.isfinite(bus_patterns["segment_m"])
    | (bus_patterns["segment_m"] < 0)
).any():
    raise ValueError("Seqment məsafəsi uyğun deyil.")

# Real dayanacaq koordinatlarını ayrıca saxlayırıq.
# 105 nömrəli dayanacaq da öz koordinatında qalır.
bus_stops_m = gpd.GeoDataFrame(
    bus_stops.copy(),
    geometry=gpd.points_from_xy(bus_stops["lon"], bus_stops["lat"]),
    crs=4326,
).to_crs(METRIC_CRS)

ordered_bus_patterns = bus_patterns.sort_values(
    ["pattern_id", "seq"]
)

pattern_summary = (
    ordered_bus_patterns
    .groupby(["route", "direction", "pattern_id"])
    .agg(
        stop_count=("stop_id", "size"),
        first_segment_m=("segment_m", "first"),
        last_segment_m=("segment_m", "last"),
        total_segment_m=("segment_m", "sum"),
    )
)

route_distance_check = (
    bus_patterns.groupby("route")["segment_m"]
    .sum().div(1000).rename("segments_cycle_km")
    .reset_index()
    .merge(
        bus_services[["route", "cycle_km_geometry"]],
        on="route",
        validate="one_to_one",
    )
)

route_distance_check["difference_m"] = (
    route_distance_check["segments_cycle_km"]
    - route_distance_check["cycle_km_geometry"]
) * 1000

print("Servis parametrləri:")
display(bus_services[[
    "route", "speed_kmh", "speed_status",
    "headway_default_min", "dwell_s", "layover_min", "coverage",
]])

print("İstiqamətlər:")
display(pattern_summary.round(3))

print("Tam dövr məsafələrinin uyğunluğu:")
display(route_distance_check.round(6))

ValueError: Seqment məsafəsi uyğun deyil.

## C3/C4 baseline runtime adapters (not C5)

The immutable c4-alpha-1 bundle is the validated computational reference. Runtime
loads saved graphs, OD access options and projected-stop shape substrings; it
never executes this notebook's analysis or server cells. Tagged cells below are
the sole source of the generated routing module. The successful projection audit
uses projected stop chainages, not the superseded source-distance slice method.
No failing bus-shape cells were present in this on-disk notebook (it ended at C3
input loading); all existing analysis is retained unchanged. Full notebook/kernel
recovery source was not available on disk; saved results are checked row by row.

Shared contracts remain unchanged. BaselineJourneyResult explicitly permits a
null overall geometry with a GEOMETRY_GAPS warning; legacy JourneyResult does not.
The legacy no_path wire time is null, while its internal objective penalty is C.
See docs/C34_ADAPTERS.md for incompatibilities and reproducible commands.


In [ ]:
from __future__ import annotations
from pathlib import Path
from typing import Any
import hashlib
import json
import math
import time
from citypilot.contracts import (RoutingRequest, ModelCatalog, EvaluationResult,
    JourneyResult, JourneyLeg, LineString, MultiLineString, Point, GeometryMetadata,
    Provenance, ODResult, ODReference, ServicePattern, ServiceGroup, Parking, ContractError)


def _check(condition, message):
    if not condition:
        raise ContractError(message)


def _sha(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024*1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


def _safe_file(root, relative):
    path = root / relative
    _check(not Path(relative).is_absolute() and path.resolve().is_relative_to(root.resolve()), 'Bundle path escapes supplied directory')
    _check(path.is_file(), 'Missing bundle file: '+relative)
    return path


def _table(root, relative):
    import pandas as pd
    return pd.read_csv(_safe_file(root, relative), low_memory=False,
        dtype={k:str for k in ('edge_id','pattern_id','service_group_id','stop_id',
               'from_stop_id','to_stop_id','route','line_id','link_id','parking_id','station_id')})


def _features(root, relative):
    data = json.loads(_safe_file(root, relative).read_text(encoding='utf-8'))
    _check(data.get('type') == 'FeatureCollection', 'Expected FeatureCollection: '+relative)
    return data['features']


def _provenance(source_id, evidence, status='assumption'):
    # Explicit source-vocabulary translation, never a wire-version inference.
    mapping = {'assumption':'assumption','assumed':'assumption','published':'published',
               'curated':'curated','observed':'observed','modelled':'modelled','unverified':'unverified'}
    _check(status in mapping, 'Unknown source evidence status: '+str(status))
    return Provenance(source_id=source_id,status=mapping[status],
        source='immutable C4 bundle',evidence=evidence)


def load_model(artifact_dir: str | Path, config_dir: str | Path) -> Any:
    """Verify and load an explicit immutable baseline bundle; never run a notebook."""
    root, config = Path(artifact_dir).resolve(), Path(config_dir).resolve()
    _check(root.is_dir() and config.is_dir(), 'Explicit artifact/config directories must exist')
    import numpy as np
    from scipy.sparse import load_npz
    manifest = json.loads((root/'manifest.json').read_text(encoding='utf-8'))
    _check(manifest.get('artifact_version') == 'c4-alpha-1', 'Unsupported bundle version')
    files = manifest['files']
    calculated = hashlib.sha256(json.dumps(files,sort_keys=True).encode('utf-8')).hexdigest()
    _check(calculated == manifest['bundle_hash'], 'Bundle hash mismatch')
    for name, digest in files.items():
        _check(_sha(_safe_file(root,name)) == digest, 'Manifest file hash mismatch: '+name)
        if name.startswith('config/'):
            _check(_sha(_safe_file(config,name.removeprefix('config/'))) == digest,
                   'External config differs from immutable bundle: '+name)
    required = {'runtime_parameters.json','nodes.csv.gz','edges.csv.gz','od/manifest.json',
        'od/rules.json','od/origins.geojson','od/destinations.geojson','od/origin_access.csv',
        'od_grid_snap_plan.geojson','metro_links.csv.gz','curated/bus_services.csv',
        'curated/bus_patterns.csv','curated/bus_stops.csv','curated/assumptions.csv',
        'bus_passenger_states.csv.gz','bus_stop_access.csv.gz','bus_stop_projections.csv.gz',
        'bus_ride_segments.csv.gz','bus_ride_geometries.geojson','metro_routes.geojson',
        'stations.geojson','metro_exit_access.csv.gz','parking_access.csv.gz'}
    required |= {f'{p}_{s}_{suffix}' for p in ('P','B') for s in ('normal','disrupted')
                 for suffix in ('time_graph.npz','arcs.csv.gz')}
    _check(required <= set(files), 'Missing required manifest entries: '+str(sorted(required-set(files))))
    runtime = json.loads((root/'runtime_parameters.json').read_text(encoding='utf-8'))
    od = json.loads((root/'od/manifest.json').read_text(encoding='utf-8'))
    rules = json.loads((root/'od/rules.json').read_text(encoding='utf-8'))
    _check(od['od_hash'] == manifest['od_hash'] and od['od_version'] == rules['version'], 'OD identity mismatch')
    _check(len(manifest['od_hash']) == 64 and all(c in '0123456789abcdef' for c in manifest['od_hash']), 'Invalid OD hash')
    origins = {f['properties']['origin_id']:f for f in _features(root,'od/origins.geojson')}
    destinations = {f['properties']['destination_id']:f for f in _features(root,'od/destinations.geojson')}
    _check(len(origins) == od['origin_count'] and len(destinations) == od['destination_count'], 'OD dimension mismatch')
    _check(rules['destination_type'] == 'individual street exit' and rules['bike_at_destination_allowed'], 'Unsupported OD semantics')
    nodes = _table(root,'nodes.csv.gz').set_index('node_id',drop=False)
    _check(nodes.index.is_unique and np.array_equal(nodes.index.to_numpy(),np.arange(len(nodes))), 'Physical node IDs must be contiguous')
    edges = _table(root,'edges.csv.gz').set_index('edge_id',drop=False)
    _check(edges.index.is_unique, 'Duplicate street edge IDs')
    _check(edges[['u','v']].min().min() >= 0 and edges[['u','v']].max().max() < len(nodes), 'Unknown street endpoint')
    access = _table(root,'od/origin_access.csv')
    _check(set(access.origin_id) == set(origins) and set(access.profile) == {'P','B'}, 'Origin/access references differ')
    _check(np.isfinite(access[['entry_walk_min','partial_edge_min','entry_min']]).all().all() and
           (access[['entry_walk_min','partial_edge_min','entry_min']] >= 0).all().all(), 'Invalid access time')
    _check(np.allclose(access.entry_min,access.entry_walk_min+access.partial_edge_min,rtol=0,atol=1e-9),'Access time breakdown mismatch')
    _check(access.physical_node_id.between(0,len(nodes)-1).all(), 'Unknown access node')
    links = _table(root,'metro_links.csv.gz')
    closures = links.loc[links.is_closure_link]
    _check(len(closures) == 1 and closures.iloc[0].normal_enabled and not closures.iloc[0].disrupted_enabled,
           'Unsupported declared closure semantics')
    model = {'root':root,'config':config,'manifest':manifest,'runtime':runtime,'od':od,'rules':rules,
        'origins':origins,'destinations':destinations,'nodes':nodes,'edges':edges,'access':access,
        'closure_id':closures.iloc[0].link_id,'graphs':{},'arc_tables':{},'solutions':{},'reverse_paths':{},
        'grid':{f['properties']['origin_id']:f['properties'] for f in _features(root,'od_grid_snap_plan.geojson')},
        'bus_geometry':{f['properties']['ride_id']:f for f in _features(root,'bus_ride_geometries.geojson')},
        'metro_geometry':{f['properties']['link_id']:f for f in _features(root,'metro_routes.geojson')},
        'stations':{f['properties']['station_id']:f for f in _features(root,'stations.geojson')},
        'services':_table(root,'curated/bus_services.csv'),'patterns':_table(root,'curated/bus_patterns.csv'),
        'stops':_table(root,'curated/bus_stops.csv'),'bus_nodes':_table(root,'bus_passenger_states.csv.gz').set_index('node_id'),
        'bus_stops':_table(root,'bus_stop_access.csv.gz').set_index('stop_id'),
        'parking':_table(root,'parking_access.csv.gz'),'metro_exits':_table(root,'metro_exit_access.csv.gz'),
        'metro_nodes':{},'links':links,
        'provenance':_provenance('baseline:'+manifest['bundle_hash'], 'Published/curated geometry; calibrated or assumed timing and access; see C34_ADAPTERS.md'),
        'warnings':tuple(w for w in manifest.get('limitations',()) if not w.startswith('Public routing adapters')) + (
            'Assumed straight grid walking/pushing access, road/metro grade transitions and parking operation; route 14 stop 105 alignment unresolved; metro entry/interchange geometry schematic.',)}
    for state in ('normal','disrupted'):
        model['metro_nodes'][state] = _table(root,f'metro_{state}_nodes.csv.gz').set_index('node_id')
    for key, meta in runtime['scenarios'].items():
        _check(key in ('P_normal','P_disrupted','B_normal','B_disrupted'), 'Unsupported saved scenario')
        graph = load_npz(root/f'{key}_time_graph.npz').tocsr()
        _check(graph.shape == (meta['node_count'],meta['node_count']), 'Matrix dimensions disagree with runtime')
        _check(graph.nnz == meta['directed_connection_count'] and graph.has_canonical_format, 'Sparse structure mismatch')
        _check(np.isfinite(graph.data).all() and (graph.data >= 0).all(), 'Invalid graph weights')
        _check(meta['node_count'] == len(nodes)+len(model['bus_nodes'])+len(model['metro_nodes'][meta['state']])+
               (len(nodes) if meta['profile']=='B' else 0), 'State node dimension mismatch')
        offset = meta['owned_offset'] or 0
        _check(offset == (runtime['scenarios']['P_'+meta['state']]['node_count'] if meta['profile']=='B' else 0), 'Invalid ownership offset')
        dests = [int(f['properties']['walk_node_id']) for f in destinations.values()]
        _check(all(0 <= v < len(nodes) and v+offset < graph.shape[0] for v in dests), 'Unknown destination node')
        _check(set(access.loc[access.profile==meta['profile'],'origin_id']) == set(origins), 'Missing profile origin access')
        model['graphs'][key]=graph
    _validate_access(model)
    return model


def _validate_access(model):
    """Check saved dual-endpoint access costs/directions, without rebuilding them."""
    for row in model['access'].itertuples(index=False):
        if row.origin_id not in model['grid']:
            _check(row.entry_min == 0, 'Non-grid origin has unexpected access cost')
            continue
        snap=model['grid'][row.origin_id];edge=model['edges'].loc[snap['edge_id']]
        node=int(row.physical_node_id)
        _check(node in (int(edge.u),int(edge.v)), 'Grid entry is not snapped segment endpoint')
        reverse=node==int(edge.u)
        # Snap-plan segments can be oriented opposite to the source edge row.
        length=float(edge.length_m)*(snap['fraction'] if node==int(snap['u']) else 1-snap['fraction'])
        can_cycle=bool(edge.bike_reverse if reverse else edge.bike_forward)
        speed=model['runtime']['c3_parameters']['walk_speed_kmh']
        mode='walk' if row.profile=='P' else 'push_bike'
        if row.profile=='B' and can_cycle and length>0:
            mode='ride_bike';speed=model['runtime']['c3_parameters']['bike_speed_kmh']
        walk=snap['snap_distance_m']/model['runtime']['c3_parameters']['walk_speed_kmh']*0.06
        _check(row.partial_edge_mode==mode and abs(row.partial_edge_min-length/speed*0.06)<1e-6 and
               abs(row.entry_walk_min-walk)<1e-6, 'Saved directional grid access disagrees with geometry/timing')


def get_catalog(model: Any) -> ModelCatalog:
    _check(isinstance(model,dict) and 'manifest' in model, 'Loaded baseline model required')
    patterns=[];groups={};sources=[];parking={}
    for pattern_id,rows in model['patterns'].groupby('pattern_id',sort=True):
        rows=rows.sort_values('seq')
        provenance=_provenance('pattern:'+pattern_id,str(rows.iloc[0]['source']),'curated');sources.append(provenance)
        patterns.append(ServicePattern(pattern_id=pattern_id,route_id=str(rows.iloc[0].route),
            direction_id=str(rows.iloc[0].direction),stop_ids=tuple(rows.stop_id),provenance=provenance))
    for row in model['services'].itertuples(index=False):
        evidence=json.dumps({k:getattr(row,k) for k in ('speed_method','speed_status','headway_status','headway_source','dwell_status','layover_status','notes')},ensure_ascii=False)
        provenance=_provenance('service:'+row.service_group_id,evidence);sources.append(provenance)
        _check(row.coverage=='full','Bundle cycle facts require explicit full coverage')
        # Geometry/counts are curated; speed, dwell and layover retain recorded
        # assumptions. SG_R2 headway is published; SG_R14 remains an assumption.
        statuses={'pattern_ids':'curated','base_headway_min':str(row.headway_status),
            'full_cycle_km':'curated','speed_kmh':str(row.speed_status),
            'stops_per_cycle':'curated','dwell_s':str(row.dwell_status),
            'layover_min':str(row.layover_status),'coverage':'curated'}
        groups[row.service_group_id]=ServiceGroup(service_group_id=row.service_group_id,
            pattern_ids=tuple(row.pattern_ids.split(';')),base_headway_min=float(row.headway_default_min),
            full_cycle_km=float(row.cycle_km_geometry),speed_kmh=float(row.speed_kmh),
            stops_per_cycle=int(row.intermediate_stops),dwell_s=float(row.dwell_s),
            layover_min=float(row.layover_min),coverage='full',field_status=statuses,provenance=provenance)
    parking_source=_provenance('parking:baseline',
        'Source-listed parking_access.csv.gz; baseline operational availability assumed for every listed site, not observed. No candidate sites supplied.')
    sources.append(parking_source)
    for parking_id in model['parking'].parking_id:
        _check(parking_id not in parking,'Duplicate parking ID')
        parking[parking_id]=Parking(parking_id=parking_id,status='assumption',baseline_enabled=True,provenance=parking_source)
    return ModelCatalog(**_identity(model),service_groups=groups,prebuilt_services={},links={},parking=parking,
        patterns=tuple(patterns),sources=tuple(sources),closure_ids=(model['closure_id'],),
        station_ids=tuple(sorted(model['stations'])),exit_ids=tuple(model['metro_exits'].access_id),
        stop_ids=tuple(model['stops'].stop_id))


def projected_bus_geometries(model):
    """Canonical successful stop-projection substring implementation, for audit.

    Saved projected chainages (NOT source chainages) locate actual shape pieces.
    Source ride lengths remain authoritative for timing. Does not write bundle.
    """
    from shapely.geometry import shape, mapping
    from shapely.ops import transform, substring
    from pyproj import Transformer
    forward=Transformer.from_crs('EPSG:4326',model['runtime']['metric_crs'],always_xy=True).transform
    backward=Transformer.from_crs(model['runtime']['metric_crs'],'EPSG:4326',always_xy=True).transform
    shapes={f['properties']['pattern_id']:transform(forward,shape(f['geometry']))
            for f in _features(model['root'],'curated/bus_route_geometry.geojson')}
    projections=_table(model['root'],'bus_stop_projections.csv.gz').set_index(['pattern_id','seq'])
    result={}
    for ride in _table(model['root'],'bus_ride_segments.csv.gz').itertuples(index=False):
        start=float(projections.loc[(ride.pattern_id,ride.from_seq)].projected_chainage_m)
        end=float(projections.loc[(ride.pattern_id,ride.to_seq)].projected_chainage_m)
        piece=substring(shapes[ride.pattern_id],start,end)
        result[ride.ride_id]=mapping(transform(backward,piece))
    return result


def _identity(model):
    # Hash-bound IDs include the immutable baseline assumptions and fixed OD set.
    return {'model_version':model['manifest']['artifact_version']+'@'+model['manifest']['bundle_hash'],
            'od_set_id':model['od']['od_version']+'@'+model['od']['od_hash']}


def _od_id(model,profile,origin_id,destination_id):
    payload=['citypilot-od-v1',_identity(model)['od_set_id'],profile,origin_id,destination_id]
    return 'od:'+profile+':'+hashlib.sha256(json.dumps(payload,ensure_ascii=False,separators=(',',':')).encode('utf-8')).hexdigest()


def get_od_table(model,profile):
    """Profile-aware stable ID mapping only; no unagreed car-distance fields."""
    _check(profile in ('P','B'),'Unknown profile')
    return tuple(ODReference(od_id=_od_id(model,profile,o,d),origin_id=o,destination_id=d,profile=profile)
        for d in model['destinations'] for o in model['origins'])


def baseline_request(model, state='normal', profile='P', thresholds_min=(40,45,50), primary_t_min=45, cap_min=120.0):
    """The immutable bundle is the complete baseline; runtime stress is unsupported."""
    return RoutingRequest(**_identity(model),state=state,
        closure_id=None if state=='normal' else model['closure_id'],profile=profile,
        thresholds_min=thresholds_min,primary_t_min=primary_t_min,cap_min=cap_min)


def _request(model, request):
    _check(isinstance(model,dict) and 'manifest' in model, 'Loaded baseline model required')
    _check(type(request) is RoutingRequest, 'A shared v1 RoutingRequest is required')
    request.to_dict()
    _check(request.state in ('normal','disrupted'), 'C5 unsupported: mitigated state')
    _check(not request.patches, 'C5 unsupported: routing patches')
    _check(request.speed_factor==1.0 and not request.headway_factor_by_group,
           'C5 unsupported: speed/headway stress; immutable baseline only')
    _check(all(getattr(request,k)==v for k,v in _identity(model).items()), 'Request model/OD identity mismatch')
    _check(request.closure_id==(None if request.state=='normal' else model['closure_id']), 'Unknown declared closure')
    return request.profile+'_'+request.state


def _status_time(value, cap):
    # Inf is permitted only inside the numerical kernel. Wire output is finite/null.
    if math.isinf(value) and value>0:
        return 'no_path',None,float(cap)
    _check(math.isfinite(value) and value>=0, 'Invalid solver time')
    return ('reachable' if value<=cap else 'over_cap'),float(value),float(min(value,cap))


def _solve(model,key):
    import numpy as np
    from scipy.sparse.csgraph import dijkstra
    if key in model['solutions']:
        return model['solutions'][key]
    meta=model['runtime']['scenarios'][key];offset=meta['owned_offset'] or 0
    targets=[];target_rows={}
    for dest_id,feature in model['destinations'].items():
        node=int(feature['properties']['walk_node_id'])
        target_rows[dest_id]=list(range(len(targets),len(targets)+(2 if offset else 1)))
        targets.append(node)
        if offset:targets.append(node+offset)
    # Reverse distances: one search per individual destination exit/ownership state.
    # Never eliminate explicit zero weights or sum parallel arc costs.
    distances,predecessors=dijkstra(model['graphs'][key].T.tocsr(),directed=True,indices=targets,return_predecessors=True)
    model['reverse_paths'][key]=(tuple(targets),predecessors)
    access=model['access'].loc[model['access'].profile==meta['profile']].reset_index(drop=True)
    result=[]
    for dest_id in model['destinations']:
        rows=target_rows[dest_id]
        for origin_id in model['origins']:
            entries=access.loc[access.origin_id==origin_id]
            starts=entries.physical_node_id.to_numpy(dtype='int64')+offset
            choices=distances[np.ix_(rows,starts)]+entries.entry_min.to_numpy()[None,:]
            target_index,entry_index=np.unravel_index(int(np.argmin(choices)),choices.shape)
            result.append({'origin_id':origin_id,'destination_id':dest_id,
                'total':float(choices[target_index,entry_index]),'entry':entries.iloc[entry_index].to_dict(),
                'start':int(starts[entry_index]),'target':targets[rows[target_index]]})
    model['solutions'][key]=result
    return result


def evaluate_package(model: Any, request: RoutingRequest) -> EvaluationResult:
    start=time.perf_counter();key=_request(model,request);rows=_solve(model,key)
    results=[]
    for row in rows:
        status,physical,penalty=_status_time(row['total'],request.cap_min)
        results.append(ODResult(od_id=_od_id(model,request.profile,row['origin_id'],row['destination_id']),
            status=status,minutes=physical,capped_minutes=penalty))
    counts={t:sum(r.minutes is not None and r.minutes<=t for r in results) for t in request.thresholds_min}
    return EvaluationResult(**_identity(model),config_hash=request.config_hash,
        state=request.state,profile=request.profile,primary_t_min=request.primary_t_min,cap_min=request.cap_min,
        od_results=tuple(results),within_counts=counts,capped_time_sum=math.fsum(r.capped_minutes for r in results),
        duration_ms=(time.perf_counter()-start)*1000,warnings=model['warnings'])


def _arcs(model,key):
    import numpy as np
    if key not in model['arc_tables']:
        arcs=_table(model['root'],key+'_arcs.csv.gz')
        # Stable min, never CSR's default parallel-edge sum. Zero arcs survive.
        arcs=arcs.sort_values('time_min',kind='stable').drop_duplicates(['u','v'],keep='first')
        graph=model['graphs'][key]
        _check(len(arcs)==graph.nnz,'Graph/arc sparsity mismatch')
        weights=np.asarray(graph[arcs.u.to_numpy(),arcs.v.to_numpy()]).ravel()
        _check(np.allclose(weights,arcs.time_min.to_numpy(),rtol=0,atol=1e-10),'Graph/arc cost mismatch')
        model['arc_tables'][key]=arcs.set_index(['u','v'])
    return model['arc_tables'][key]


def _minimum_graph(arcs, node_count):
    """Small-case and future loader helper: preserve zero edges, minimum parallels."""
    from scipy.sparse import csr_matrix
    cheapest={}
    for u,v,cost in arcs:
        _check(math.isfinite(cost) and cost>=0 and 0<=u<node_count and 0<=v<node_count,'Invalid arc')
        cheapest[u,v]=min(cheapest.get((u,v),math.inf),cost)
    keys=sorted(cheapest)
    return csr_matrix(([cheapest[k] for k in keys],([k[0] for k in keys],[k[1] for k in keys])),shape=(node_count,node_count))


def _xy_to_lonlat(model,xy):
    from pyproj import Transformer
    if 'transformer' not in model:
        model['transformer']=Transformer.from_crs(model['runtime']['metric_crs'],'EPSG:4326',always_xy=True)
    return tuple(float(v) for v in model['transformer'].transform(*xy))


def _node_coord(model,key,node):
    meta=model['runtime']['scenarios'][key];offset=meta['owned_offset'] or 0
    if offset and node>=offset:node-=offset
    if node in model['nodes'].index:
        row=model['nodes'].loc[node];return _xy_to_lonlat(model,(row.x,row.y))
    if node in model['bus_nodes'].index:
        stop_id=model['bus_nodes'].loc[node].stop_id;row=model['bus_stops'].loc[stop_id]
        return _xy_to_lonlat(model,(row.source_x,row.source_y))
    station_id=model['metro_nodes'][meta['state']].loc[node].station_id
    return tuple(model['stations'][station_id]['geometry']['coordinates'])


def _path_geometry(coords):
    points=tuple(tuple(map(float,c)) for c in coords)
    _check(bool(points),'Empty physical geometry')
    return Point(coordinates=points[0]) if all(p==points[0] for p in points) else LineString(coordinates=points)


def _assemble_geometry(legs,provenance):
    """Preserve every piece. Exact endpoint equality only; never bridge a gap."""
    pieces=[];gaps=0
    for leg in legs:
        geometry=leg.geometry
        lines=(geometry.coordinates,) if isinstance(geometry,MultiLineString) else (
            (geometry.coordinates,) if isinstance(geometry,Point) else geometry.coordinates,)
        for line in lines:
            points=list(line)
            if pieces and pieces[-1][-1]==points[0]:
                pieces[-1].extend(points[1:])
            else:
                if pieces:gaps+=1
                pieces.append(points)
    _check(bool(pieces),'Existing journey has no geometry')
    if gaps:
        geometry=MultiLineString(coordinates=tuple(tuple(p if len(p)>1 else p*2) for p in pieces))
        note=_provenance(provenance.source_id+':geometry-gaps',
            'Partial source geometry: '+str(gaps)+' disconnected boundaries retained; no inferred connecting segments.')
        metadata=GeometryMetadata(partial=True,gap_count=gaps,provenance=(provenance,note))
    else:
        geometry=_path_geometry(pieces[0])
        metadata=GeometryMetadata(partial=False,gap_count=0,provenance=(provenance,))
    return geometry,metadata


def _leg(mode,start,end,geometry,minutes,component,provenance,route=None,pattern=None,board=None,alight=None):
    components={'walk_min':0.0,'wait_min':0.0,'ride_min':0.0,'transfer_min':0.0};components[component]=float(minutes)
    return JourneyLeg(mode=mode,from_id=start,to_id=end,route_id=route,pattern_id=pattern,
        board_stop_id=board,alight_stop_id=alight,geometry=_path_geometry(geometry),
        provenance=(provenance,),**components)


def _access_legs(model,key,row):
    entry=row['entry'];origin=row['origin_id'];start='state:'+str(row['start']);legs=[]
    source=tuple(model['origins'][origin]['geometry']['coordinates'])
    if origin in model['grid']:
        snap=model['grid'][origin];point=_xy_to_lonlat(model,(snap['snap_x'],snap['snap_y']))
        middle='grid_snap:'+origin
        legs.append(_leg('walk',origin,middle,(source,point),entry['entry_walk_min'],'walk_min',model['provenance']))
        endpoint=_node_coord(model,key,row['start'])
        cycling=entry['partial_edge_mode']=='ride_bike'
        legs.append(_leg('bike' if cycling else 'walk',middle,start,(point,endpoint),
            entry['partial_edge_min'],'ride_min' if cycling else 'walk_min',model['provenance']))
    else:
        point=_node_coord(model,key,row['start'])
        legs.append(_leg('walk',origin,start,(point,point),0,'walk_min',model['provenance']))
    return legs


def _arc_geometry(model,key,u,v,arc):
    kind=arc['kind'];a=_node_coord(model,key,u);b=_node_coord(model,key,v)
    if kind=='bus_ride':
        return model['bus_geometry'][arc['edge_id']]['geometry']['coordinates']
    if kind=='metro_ride':
        feature=model['metro_geometry'][arc['link_id']];props=feature['properties']
        coords=feature['geometry']['coordinates']
        station=model['metro_nodes'][model['runtime']['scenarios'][key]['state']].loc[u].station_id
        link=model['links'].set_index('link_id').loc[arc['link_id']]
        return coords if station==link.from_station_id else list(reversed(coords))
    if kind in ('bus_board','bus_alight','bus_continue','metro_board','metro_alight','metro_continue','park_bike'):
        return (a,a)  # zero-motion/wait state transition, never a fabricated physical link
    return (a,b)  # actual street segment or explicitly schematic metro access/interchange


def _journey_legs(model,key,row,path):
    legs=_access_legs(model,key,row);arcs=_arcs(model,key);parking_count=0
    offset=model['runtime']['scenarios'][key]['owned_offset'] or 0
    state=model['runtime']['scenarios'][key]['state']
    for u,v in zip(path,path[1:]):
        arc=arcs.loc[(u,v)];kind=arc['kind'];owned=bool(offset and u>=offset)
        if kind=='park_bike':
            parking_count+=1;_check(owned and v<offset and parking_count<=1,'Invalid parking ownership transition')
        elif owned:
            _check(v>=offset and kind in ('ride_bike','push_bike'), 'Transit boarding with bike or invalid retrieval')
        else:
            _check(not offset or v<offset,'Bike retrieval forbidden')
        geometry=_arc_geometry(model,key,u,v,arc)
        mode='walk';component='walk_min';route=pattern=board=alight=None
        if kind.startswith('bus_'):
            mode='bus';pattern=str(arc['pattern_id'])
            node=u if u in model['bus_nodes'].index else v
            meta=model['bus_nodes'].loc[node];route=str(meta.route);board=str(meta.stop_id);alight=board
            if kind=='bus_ride':alight=str(model['bus_nodes'].loc[v].stop_id)
            component='wait_min' if kind=='bus_board' else 'ride_min'
        elif kind in ('metro_ride','metro_board','metro_alight','metro_continue'):
            mode='metro';node=u if u in model['metro_nodes'][state].index else v
            meta=model['metro_nodes'][state].loc[node];route=str(arc['line_id']);pattern='metro:'+route+':'+str(meta.direction)
            board=str(meta.station_id);alight=str(model['metro_nodes'][state].loc[v].station_id)
            component='wait_min' if kind=='metro_board' else 'ride_min'
        elif kind=='ride_bike':mode='bike';component='ride_min'
        elif kind in ('park_bike','metro_enter','metro_exit','metro_interchange'):mode='transfer';component='transfer_min'
        legs.append(_leg(mode,'state:'+str(u),'state:'+str(v),geometry,arc['time_min'],component,
                         model['provenance'],route,pattern,board,alight))
    end=_node_coord(model,key,row['target'])
    legs.append(_leg('walk','state:'+str(row['target']),row['destination_id'],(end,end),0,'walk_min',model['provenance']))
    return tuple(legs)


def inspect_journey(model: Any, request: RoutingRequest, origin_id: str, destination_id: str) -> JourneyResult:
    key=_request(model,request)
    _check(origin_id in model['origins'] and destination_id in model['destinations'],'Unknown saved origin/destination')
    row=next(r for r in _solve(model,key) if r['origin_id']==origin_id and r['destination_id']==destination_id)
    status,physical,penalty=_status_time(row['total'],request.cap_min)
    identity=dict(_identity(model),config_hash=request.config_hash,state=request.state,profile=request.profile,
        od_id=_od_id(model,request.profile,origin_id,destination_id),origin_id=origin_id,destination_id=destination_id,
        status=status,cap_min=request.cap_min,total_min=physical,provenance=(model['provenance'],))
    if status=='no_path':
        return JourneyResult(**identity,geometry=None,legs=(),warnings=model['warnings'],
            geometry_metadata=GeometryMetadata(partial=False,gap_count=0,provenance=(model['provenance'],)))
    targets,predecessors=model['reverse_paths'][key]
    predecessor=predecessors[targets.index(row['target'])]
    path=[row['start']];seen={path[0]}
    while path[-1]!=row['target']:
        node=int(predecessor[path[-1]])
        _check(node>=0 and node not in seen,'Broken/cyclic directed predecessor chain')
        path.append(node);seen.add(node)
    legs=_journey_legs(model,key,row,path)
    _check(abs(math.fsum(l.total_min for l in legs)-physical)<1e-6,'Reconstructed route differs from reverse OD time')
    geometry,metadata=_assemble_geometry(legs,model['provenance'])
    warnings=model['warnings']+(('GEOMETRY_GAPS',) if metadata.partial else ())
    return JourneyResult(**identity,geometry=geometry,geometry_metadata=metadata,legs=legs,
        generalised_cost_min=physical,warnings=warnings)
